<div style="width: 96ch;">
<h2>Exoskelet van een knie (&ldquo;<a href="https://www.mdpi.com/2227-7080/13/10/463">eight-bar linkage</a>&rdquo;)</h2>

<div style="display: flex; gap: 25px; align-items: flex-start;">
  <figure style="width: 38%; text-align: center;">
    <img src="Images/mechanisme_paper.jpg" style="width: 100%;">
    <figcaption><b>Figuur:</b> Mechanisme uit het verslag.</figcaption>
  </figure>

  <figure style="width: 58%; text-align: center;">
    <img src="Images\wandelen_paper.jpg" style="width: 100%;">
    <figcaption><b>Figuur:</b> Serie van fotos waarop wandelpatroon van mechanisme te zien is.</figcaption>
  </figure>
</div>

### Mobiliteit in het vlak. 
Het aantal vrijheidsgraden kan bepaald worden met de formule van Grübler: 
$$M = 3*(n-1) - 2*f_1 - f_2 = 1 \,\text{DOF}$$
$n = 8$ <br>
$f_1 = 7 + 1 + 1 + 1 = 10$ *(gewricht A, D, E nemen 2 keer 2 vrijheidsgraden weg)* <br>
$f_2 = 0$ <br>

Het resultaat is een mechanisme met 1 vrijheidsgraad. De posities en orientaties van elke stang is dus afhankelijk van enkel $\theta_2$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

### Variabelen
- De snelheid waarmee gewandelt wordt kan omgezet worden in een hoeksnelheid via de gemiddelde staplengte van een persoon. Dit is gelijk aan 1.44m/cyclus. Hieruit volgt dat:

$ \omega = v*2*\pi/l $ met $l$ gelijk aan de gemiddelde afgelegde horizontale afstand per cyclus

In [ ]:
# design variables
r_1 = 0.28  # fixed link length, distance between O_1 and O_2
r_2 = 0.08 # fixed length between O_1 and A, connected to the driver
r_3 = 0.28# fixed length between A and between B
r_4 = 0.427# fixed length between B and O_2
r_DC = 0.17 # fixed distance between D and C
r_6 = 0.46 # fixed distance between A and E
r_7 = 0.52 # fixed distance between C and F
r_8 = 0.06 # fixed distance between E and F
theta_0 = 19.7 * np.pi/180 # angle between two fixed points
theta_C = 128 * np.pi/180 # angle between BDC
theta_I = 85 * np.pi/180  # angle of FEG
L_1 = 0.46 # length of the leg: DE
L_2 = 0.4 # length of the leg: EG

"""
Initial conditions for the angles of the links:
"""
theta_3_init = 275 * np.pi/180
theta_4_init = 260 * np.pi/180
theta_6_init = 285 * np.pi/180
theta_7_init = 275 * np.pi/180
theta_8_init = -30 * np.pi/180
theta_f_init = 270 * np.pi/180

t_begin =  0     # start time of simulat
t_end   = 10     # end time of simulation
Ts      =  0.01  # time step of simulation
t = np.arange(t_begin, t_end, Ts)  # time vector


# initialization of driver
omega = - 2.1      # driver frequency
A     = 1        # amplitude
# angular position, velocity and acceleration:
theta_2   = omega * t   #0 + A * np.sin(omega * t)
dtheta_2  = np.ones_like(t) * omega  #* A * np.cos(omega * t)
ddtheta_2 = np.zeros_like(t) #-omega ** 2 * A * np.sin(omega * t)

# Kinematische analyse

<div style="display:flex; align-items:flex-start; gap:30px;">

<div style="flex:1.4;">

We hebben drie verschillende gesloten kinematische lussen (vier stangen die altijd met elkaar verbonden zijn). Dit geeft ons de drie sluitingsvergelijkingen hieronder. Elke vector heeft een x en y-coördinaat dus dit geeft ons 6 vergelijkingen en 6 onbekenden. Het stelsel van vergelijkingen is niet-lineair dus we gebruiken een numerieke solver.

1. $\overrightarrow{0_2A} + \overrightarrow{AB} + \overrightarrow{BD} + \overrightarrow{DA} = 0$

2. $\overrightarrow{0_2A} + \overrightarrow{AE} + \overrightarrow{ED} + \overrightarrow{DA} = 0$

3. $\overrightarrow{DE} + \overrightarrow{EF} + \overrightarrow{FC} + \overrightarrow{CD} = 0$

</div>

<div style="flex:1; text-align:center;">

<img src="Images\sluitingsvgl.png" width="500"><br>
<b>Figure 1:</b> Kinematische lussen van het mechanisme.

</div>

</div>

In [ ]:
"""
Sluitingsvergelijkingen voor de drie lussen:
"""

# loop closure equations for the first loop (O_2-A-B-D)
def loop_closure1(theta_init, theta_2, r_1, r_2, r_3, r_4, theta_0):
    theta_3 = theta_init[0]
    theta_4 = theta_init[1]

    eq1 = r_2 * np.cos(theta_2) + r_3 * np.cos(theta_3) - r_4 * np.cos(theta_4) - r_1 * np.cos(theta_0)
    eq2 = r_2 * np.sin(theta_2) + r_3 * np.sin(theta_3) - r_4 * np.sin(theta_4) - r_1 * np.sin(theta_0)

    return [eq1, eq2]

# loop closure equations for the second loop (O_2-A-E-D)
def loop_closure2(theta_init, theta_2, r_1, r_2, r_6, L_1, theta_0):
    theta_6 = theta_init[0]
    theta_f = theta_init[1]

    eq1 = r_2 * np.cos(theta_2) + r_6 * np.cos(theta_6) - L_1 * np.cos(theta_f) - r_1 * np.cos(theta_0)
    eq2 = r_2 * np.sin(theta_2) + r_6 * np.sin(theta_6) - L_1 * np.sin(theta_f) - r_1 * np.sin(theta_0)

    return [eq1, eq2]

#loop closure equations for the third loop (D-E-F-C)
def loop_closure3(theta_init, theta_4, theta_f, theta_C, L_1, r_7, r_8, r_DC):
    theta_7 = theta_init[0]
    theta_8 = theta_init[1]

    eq1 = r_DC*np.cos(theta_C + theta_4) + r_7 * np.cos(theta_7) - r_8 * np.cos(theta_8) - L_1 * np.cos(theta_f)
    eq2 = r_DC*np.sin(theta_C + theta_4) + r_7 * np.sin(theta_7) - r_8 * np.sin(theta_8) - L_1 * np.sin(theta_f)

    return [eq1, eq2]



In [ ]:
# Function to rotate a vector z over an angle theta:
def rotate_vector(z, theta):
    rotation_matrix = np.array([[np.cos(theta), -np.sin(theta)],
                                [np.sin(theta), np.cos(theta)]])
    return np.dot(rotation_matrix, z)

**Werking functie kinematics8bar:** De functie berekent de kinematica van het volledige 8-bar kniemechanisme. Voor elke tijdstap worden de onbekende hoeken bepaald via sluitingsvergelijkingen. Daarna worden de overeenkomstige hoeksnelheden en hoekversnellingen berekend door de snelheids- en versnellingsvergelijkingen als lineaire stelsels op te lossen.

Niet lineaire fsolve werkt met Newton-Raphson (alleen dan met matrices): $x_{new} = x_{old} - \frac{f(x_{old})}{f'(x_{old})}$

**Snelheidsvergelijkingen:** Snelheidsvergelijkingen worden bekomen uit de tijdsafgeleide van de positie-sluitingsvergelijkingen

bv. x-richting van loop closure lus 1 wordt:

$\frac{d}{dt}(r_2\cdot cos(\theta_2) + r_3\cdot cos(\theta_3) - r_4\cdot cos(\theta_4) - r_1\cdot cos(\theta_0))$

$=-r_2\cdot sin(\theta_2)\omega_2 - r_3\cdot sin(\theta_3)\omega_3 + r_4\cdot sin(\theta_4)\omega_4 + r_1\cdot sin(\theta_0)\omega_0$

**Versnellingsvergelijkingen:** Op dezelfde manier wordt voor de versnelling nog eens afgeleid naar de tijd. Omdat de snelheids en versnellingsvergelijkingen lineaire stelsels zijn kunnen ze gewoon met matrices opgelost worden.

In [ ]:
def kinematics8bar(r_1, r_2, r_3, r_4, r_DC, r_6, r_7, r_8, theta_0, theta_C, theta_I, theta_2, dtheta_2, ddtheta_2, theta_3_init, theta_4_init, theta_6_init, theta_7_init, theta_f_init, theta_8_init, t):
    global theta_3, theta_4, theta_6, theta_f, theta_7, theta_8, dtheta_3, dtheta_4, dtheta_6, dtheta_f, dtheta_7, dtheta_8, ddtheta_3, ddtheta_4, ddtheta_6, ddtheta_f, ddtheta_7, ddtheta_8, cond
    theta_3 = np.zeros_like(theta_2)
    theta_4 = np.zeros_like(theta_2)
    theta_6 = np.zeros_like(theta_2)
    theta_f = np.zeros_like(theta_2)
    theta_7 = np.zeros_like(theta_2)
    theta_8 = np.zeros_like(theta_2)

    dtheta_3 = np.zeros_like(theta_2)
    dtheta_4 = np.zeros_like(theta_2)
    dtheta_6 = np.zeros_like(theta_2)
    dtheta_f = np.zeros_like(theta_2)
    dtheta_7 = np.zeros_like(theta_2)
    dtheta_8 = np.zeros_like(theta_2)

    ddtheta_3 = np.zeros_like(theta_2)
    ddtheta_4 = np.zeros_like(theta_2)
    ddtheta_6 = np.zeros_like(theta_2)
    ddtheta_f = np.zeros_like(theta_2)
    ddtheta_7 = np.zeros_like(theta_2)
    ddtheta_8 = np.zeros_like(theta_2)

    cond   = np.zeros((len(t),3))

    optim_options = {"full_output":True}  # options for fsolve

    for k, time in enumerate(t):

    # Loop closure analysis for the first loop
        x1, _, ier, message  = fsolve(lambda x1: loop_closure1(x1, theta_2[k], r_1, r_2, r_3, r_4, theta_0), [theta_3_init, theta_4_init], **optim_options)

        if ier != 1:
            print("The fsolve exit flag was not 1, probably no convergence!")
            print(message)

        theta_3[k] = x1[0]
        theta_4[k] = x1[1]

          
        # Velocity Analysis
        A1 = np.array([[-r_4 * np.sin(theta_4[k]), r_3 * np.sin(theta_3[k])],
                    [r_4 * np.cos(theta_4[k]), -r_3 * np.cos(theta_3[k])]])
        B1 = np.array([- r_2 * np.sin(theta_2[k]) * dtheta_2[k],
                        r_2 * np.cos(theta_2[k]) * dtheta_2[k]])

        (dtheta_4[k], dtheta_3[k]) = np.linalg.solve(A1, B1)
        cond[k,0] = np.linalg.cond(A1)

        #Acceleration analysis for the first loop
        C1 = np.array([r_2*(-np.sin(theta_2[k]) * ddtheta_2[k] - np.cos(theta_2[k]) * dtheta_2[k]**2) - r_3*np.cos(theta_3[k]) * dtheta_3[k]**2 + r_4* np.cos(theta_4[k]) * dtheta_4[k]**2,
            r_2*(np.cos(theta_2[k]) * ddtheta_2[k] - np.sin(theta_2[k]) * dtheta_2[k]**2) - r_3*np.sin(theta_3[k]) * dtheta_3[k]**2 + r_4* np.sin(theta_4[k]) * dtheta_4[k]**2])

        (ddtheta_4[k], ddtheta_3[k]) = np.linalg.solve(A1, C1)

    # Loop closure analysis for the second loop
        x2, _, ier, message  = fsolve(lambda x2: loop_closure2(x2, theta_2[k], r_1, r_2, r_6, L_1, theta_0), [theta_6_init, theta_f_init], **optim_options)


        if ier != 1:
            print("The fsolve exit flag was not 1, probably no convergence!")
            print(message)

        theta_6[k] = x2[0]
        theta_f[k] = x2[1]



        # Velocity Analysis
        A2 = np.array([[-L_1 * np.sin(theta_f[k]), r_6 * np.sin(theta_6[k])],
                    [L_1 * np.cos(theta_f[k]), -r_6 * np.cos(theta_6[k])]])
        B2 = np.array([- r_2 * np.sin(theta_2[k]) * dtheta_2[k],
                        r_2 * np.cos(theta_2[k]) * dtheta_2[k]])

        (dtheta_f[k], dtheta_6[k]) = np.linalg.solve(A2, B2)
        cond[k,1] = np.linalg.cond(A2)

        #Acceleration analysis for the second loop
        C2 = np.array([r_2*(-np.sin(theta_2[k]) * ddtheta_2[k] - np.cos(theta_2[k]) * dtheta_2[k]**2) - r_6*np.cos(theta_6[k]) * dtheta_6[k]**2 + L_1* np.cos(theta_f[k]) * dtheta_f[k]**2,
            r_2*(np.cos(theta_2[k]) * ddtheta_2[k] - np.sin(theta_2[k]) * dtheta_2[k]**2) - r_6*np.sin(theta_6[k]) * dtheta_6[k]**2 + L_1* np.sin(theta_f[k]) * dtheta_f[k]**2])

        (ddtheta_f[k], ddtheta_6[k]) = np.linalg.solve(A2, C2)
    
    # Loop closure analysis for the third loop
        x3, _, ier, message = fsolve(lambda x3: loop_closure3(x3, theta_4[k], theta_f[k], theta_C, L_1, r_7, r_8, r_DC), [theta_7_init, theta_8_init], **optim_options)

        if ier != 1:
            print("The fsolve exit flag was not 1, probably no convergence!")
            print(message)

        theta_7[k] = x3[0]
        theta_8[k] = x3[1]

        # Velocity Analysis
        A3 = np.array([[r_7* np.sin(theta_7[k]), -r_8 * np.sin(theta_8[k])],
                    [-r_7 * np.cos(theta_7[k]),  r_8 * np.cos(theta_8[k])]])
        B3 = np.array([- r_DC * np.sin(theta_4[k] + theta_C) * dtheta_4[k] + L_1 * np.sin(theta_f[k]) * dtheta_f[k],
                        r_DC * np.cos(theta_4[k] + theta_C) * dtheta_4[k] -L_1* np.cos(theta_f[k]) * dtheta_f[k]])

        (dtheta_7[k], dtheta_8[k]) = np.linalg.solve(A3, B3)
        cond[k,2] = np.linalg.cond(A3)

        # Acceleration Analysis for the third loop
        C3 = np.array([- r_DC * (np.cos(theta_4[k] + theta_C) * dtheta_4[k]**2 + np.sin(theta_4[k] + theta_C) * ddtheta_4[k]) + L_1 * (np.sin(theta_f[k]) * ddtheta_f[k] + np.cos(theta_f[k]) * dtheta_f[k]**2) - r_7*np.cos(theta_7[k]) * dtheta_7[k]**2 + r_8 * np.cos(theta_8[k]) * dtheta_8[k]**2,
                        r_DC * (-np.sin(theta_4[k] + theta_C) * dtheta_4[k]**2 + np.cos(theta_4[k] + theta_C) * ddtheta_4[k]) - L_1 * (np.cos(theta_f[k]) * ddtheta_f[k] - np.sin(theta_f[k]) * dtheta_f[k]**2) - r_7*np.sin(theta_7[k]) * dtheta_7[k]**2 + r_8 * np.sin(theta_8[k]) * dtheta_8[k]**2])

        (ddtheta_7[k], ddtheta_8[k]) = np.linalg.solve(A3, C3)

        # Next iteration step
        theta_3_init = theta_3[k] + (t[1] - t[0]) * dtheta_3[k]
        theta_4_init = theta_4[k] + (t[1] - t[0]) * dtheta_4[k]

        theta_6_init = theta_6[k] + (t[1] - t[0]) * dtheta_6[k]
        theta_f_init = theta_f[k] + (t[1] - t[0]) * dtheta_f[k]

        theta_7_init = theta_7[k] + (t[1] - t[0]) * dtheta_7[k]
        theta_8_init = theta_8[k] + (t[1] - t[0]) * dtheta_8[k]
    

In [ ]:
# Animation of mechanism motion
global fig, ax, x_left, x_right, y_bottom, y_top, O_2, D, index_vec
# compute width and height of drawing canvas:
x_left   = -5 * r_2
y_bottom = -1.25 * max(r_2, r_4)
x_right  = r_2 + r_6 + 1.25 * L_2
y_top    = r_2 + r_6 + 1.25 * L_2

# Don't show figure until drawn explicitly:
plt.ioff()
# Initialize the figure
fig, ax = plt.subplots()

# Define the function to update the plot for each frame:
t_size = len(t)
# fraction of simulated frames in the animation:
frames = t_size/2

# Define point O_2 and D as ground link:
O_2 = np.array([0,0.5])
D = O_2 + rotate_vector(np.array([r_1, 0]), theta_0)

# time between frames
delta = int(np.floor(t_size / frames))
# index vector for frames        
index_vec = np.arange(0, t_size, delta)

# calculation and visualisation of the kinematics:
kinematics8bar(r_1, r_2, r_3, r_4, r_DC, r_6, r_7, r_8, theta_0, theta_C, theta_I, theta_2, dtheta_2, ddtheta_2, theta_3_init, theta_4_init, theta_6_init, theta_7_init, theta_f_init, theta_8_init, t)

# next step in the animation:
def update(frame):
    ax.cla()  # Clear the current axes -> every previous frame is cleared before drawing the next frame
    ax.axis('equal')  # Ensures 1 unit on x == 1 unit on y
    ax.set_xlabel('[m]')
    ax.set_ylabel('[m]')
    ax.set_xlim([x_left, x_right])
    ax.set_ylim([y_bottom, y_top])
    ax.set_title(f'Frame {frame}')

    # Calculate Cartesian coordinates using rotation matrix
    A = O_2 + rotate_vector(np.array([r_2, 0]), theta_2[frame])
    B1 = A + rotate_vector(np.array([r_3, 0]), theta_3[frame])
    B2 = D + rotate_vector(np.array([r_4, 0]), theta_4[frame])
    loop1 = np.array([O_2, A, B1, B2, D])
    ax.plot(loop1[:, 0], loop1[:, 1], '-o')

    A = O_2 + rotate_vector(np.array([r_2, 0]), theta_2[frame])
    E1 = A + rotate_vector(np.array([r_6, 0]), theta_6[frame])
    E2 = D + rotate_vector(np.array([L_1, 0]), theta_f[frame])
    loop2 = np.array([O_2, A, E1, E2, D])
    ax.plot(loop2[:, 0], loop2[:, 1], '-o')

    E = D + rotate_vector(np.array([L_1, 0]), theta_f[frame])
    F1 = E + rotate_vector(np.array([r_8, 0]), theta_8[frame])
    C = D + rotate_vector(np.array([r_DC, 0]), theta_4[frame] + theta_C)
    F2 = C + rotate_vector(np.array([r_7, 0]), theta_7[frame])
    loop3 = np.array([D, E, F1, F2, C, D])
    ax.plot(loop3[:, 0], loop3[:, 1], '-o')

    B = D + rotate_vector(np.array([r_4, 0]), theta_4[frame])
    loop4 = np.array([D,B,C,D])
    ax.plot(loop4[:, 0], loop4[:, 1], '-o')

    G = E + rotate_vector(np.array([L_2, 0]),  -(theta_I - theta_8[frame]))
    F = E + rotate_vector(np.array([r_8,0]), theta_8[frame])
    loop5 = np.array([E, F , G, E])
    ax.plot(loop5[:, 0], loop5[:, 1], '-o')
    return(ax)

# Update the animation
ani = FuncAnimation(fig, update, frames=len(index_vec), interval=50)


In [ ]:

# calculation and visualisation of the kinematics:
kinematics8bar(r_1, r_2, r_3, r_4, r_DC, r_6, r_7, r_8, theta_0, theta_C, theta_I, theta_2, dtheta_2, ddtheta_2, theta_3_init, theta_4_init, theta_6_init, theta_7_init, theta_f_init, theta_8_init, t)

# next step in the animation:
def update(frame):
    ax.cla()  # Clear the current axes -> every previous frame is cleared before drawing the next frame
    ax.axis('equal')  # Ensures 1 unit on x == 1 unit on y
    ax.set_xlabel('[m]')
    ax.set_ylabel('[m]')
    ax.set_xlim([x_left, x_right])
    ax.set_ylim([y_bottom, y_top])
    ax.set_title(f'Frame {frame}')

    # Calculate Cartesian coordinates using rotation matrix
    A = O_2 + rotate_vector(np.array([r_2, 0]), theta_2[frame])
    B1 = A + rotate_vector(np.array([r_3, 0]), theta_3[frame])
    B2 = D + rotate_vector(np.array([r_4, 0]), theta_4[frame])
    loop1 = np.array([O_2, A, B1, B2, D])
    ax.plot(loop1[:, 0], loop1[:, 1], '-o')

    A = O_2 + rotate_vector(np.array([r_2, 0]), theta_2[frame])
    E1 = A + rotate_vector(np.array([r_6, 0]), theta_6[frame])
    E2 = D + rotate_vector(np.array([L_1, 0]), theta_f[frame])
    loop2 = np.array([O_2, A, E1, E2, D])
    ax.plot(loop2[:, 0], loop2[:, 1], '-o')

    E = D + rotate_vector(np.array([L_1, 0]), theta_f[frame])
    F1 = E + rotate_vector(np.array([r_8, 0]), theta_8[frame])
    C = D + rotate_vector(np.array([r_DC, 0]), theta_4[frame] + theta_C)
    F2 = C + rotate_vector(np.array([r_7, 0]), theta_7[frame])
    loop3 = np.array([D, E, F1, F2, C, D])
    ax.plot(loop3[:, 0], loop3[:, 1], '-o')

    B = D + rotate_vector(np.array([r_4, 0]), theta_4[frame])
    loop4 = np.array([D,B,C,D])
    ax.plot(loop4[:, 0], loop4[:, 1], '-o')

    G = E + rotate_vector(np.array([L_2, 0]),  -(theta_I - theta_8[frame]))
    F = E + rotate_vector(np.array([r_8,0]), theta_8[frame])
    loop5 = np.array([E, F , G, E])
    ax.plot(loop5[:, 0], loop5[:, 1], '-o')
    return(ax)

# Update the animation
ani = FuncAnimation(fig, update, frames=len(index_vec), interval=50)


In [ ]:
# ------------------------------------------------------------
# Volledig pad van punt G over één cyclus berekenen
# ------------------------------------------------------------

omega_driver = np.mean(np.abs(dtheta_2))
i_end_cycle = int(np.round((2*np.pi / omega_driver) / Ts))

# Gebruik een lokale vaste versie van punt D
# Zo vermijden we dat een andere variabele D uit de notebook problemen geeft.
D_path = O_2 + rotate_vector(np.array([r_1, 0]), theta_0)

G_path = []

for i in range(i_end_cycle):
    E_i = D_path + rotate_vector(np.array([L_1, 0]), theta_f[i])
    G_i = E_i + rotate_vector(np.array([L_2, 0]), -(theta_I - theta_8[i]))
    G_path.append(G_i)

G_path = np.array(G_path)


# next step in the animation:
def update(frame):
    ax.cla()  # Clear the current axes -> every previous frame is cleared before drawing the next frame
    ax.axis('equal')  # Ensures 1 unit on x == 1 unit on y
    ax.set_xlabel('[m]')
    ax.set_ylabel('[m]')
    ax.set_xlim([x_left, x_right])
    ax.set_ylim([y_bottom, y_top])
    ax.set_title(f'Frame {frame}')
    # Pad van punt G tekenen
    ax.plot(G_path[:, 0], G_path[:, 1], color='blue', linewidth=0.8, alpha=0.6)

    # Calculate Cartesian coordinates using rotation matrix
    A = O_2 + rotate_vector(np.array([r_2, 0]), theta_2[frame])
    B1 = A + rotate_vector(np.array([r_3, 0]), theta_3[frame])
    B2 = D + rotate_vector(np.array([r_4, 0]), theta_4[frame])
    loop1 = np.array([O_2, A, B1, B2, D])
    ax.plot(loop1[:, 0], loop1[:, 1], '-o')

    A = O_2 + rotate_vector(np.array([r_2, 0]), theta_2[frame])
    E1 = A + rotate_vector(np.array([r_6, 0]), theta_6[frame])
    E2 = D + rotate_vector(np.array([L_1, 0]), theta_f[frame])
    loop2 = np.array([O_2, A, E1, E2, D])
    ax.plot(loop2[:, 0], loop2[:, 1], '-o')

    E = D + rotate_vector(np.array([L_1, 0]), theta_f[frame])
    F1 = E + rotate_vector(np.array([r_8, 0]), theta_8[frame])
    C = D + rotate_vector(np.array([r_DC, 0]), theta_4[frame] + theta_C)
    F2 = C + rotate_vector(np.array([r_7, 0]), theta_7[frame])
    loop3 = np.array([D, E, F1, F2, C, D])
    ax.plot(loop3[:, 0], loop3[:, 1], '-o')

    B = D + rotate_vector(np.array([r_4, 0]), theta_4[frame])
    loop4 = np.array([D,B,C,D])
    ax.plot(loop4[:, 0], loop4[:, 1], '-o')

    G = E + rotate_vector(np.array([L_2, 0]),  -(theta_I - theta_8[frame]))
    F = E + rotate_vector(np.array([r_8,0]), theta_8[frame])
    loop5 = np.array([E, F , G, E])
    ax.plot(loop5[:, 0], loop5[:, 1], '-o')
    return(ax)

# Update the animation
ani = FuncAnimation(fig, update, frames=len(index_vec), interval=50)


In [ ]:
# display mechanism animation:
ani_html = ani.to_jshtml(default_mode='once')
HTML(ani_html)

In [ ]:
"""
Hier definieren we eerst de punten van het mechanisme als functie van de tijd
"""


# Punten van het mechanisme als functie van de tijd
# Elke rij is [x, y, z] voor één tijdstip i

O_2_array = np.zeros((len(t), 3))
A = np.zeros((len(t), 3))
D = np.zeros((len(t), 3))
B = np.zeros((len(t), 3))
C = np.zeros((len(t), 3))
E = np.zeros((len(t), 3))
F = np.zeros((len(t), 3))
G = np.zeros((len(t), 3))

# Vast punt O2
O_2_fixed = np.array([0.0, 0.0, 0.0]) #waarom niet als oorsprong?

for i in range(len(t)):
    O_2_array[i] = O_2_fixed

    # Vast punt D, gemeten vanaf O2
    D[i] = O_2_fixed + np.array([
        r_1 * np.cos(theta_0),
        r_1 * np.sin(theta_0),
        0.0
    ])

    # Punt A op link r2
    A[i] = O_2_fixed + np.array([
        r_2 * np.cos(theta_2[i]),
        r_2 * np.sin(theta_2[i]),
        0.0
    ])

    # Punt B op de driehoek DBC
    B[i] = D[i] + np.array([
        r_4 * np.cos(theta_4[i]),
        r_4 * np.sin(theta_4[i]),
        0.0
    ])

    # Punt C op de driehoek DBC
    C[i] = D[i] + np.array([
        r_DC * np.cos(theta_4[i] + theta_C),
        r_DC * np.sin(theta_4[i] + theta_C),
        0.0
    ])

    # Punt E op staaf DE
    E[i] = D[i] + np.array([
        L_1 * np.cos(theta_f[i]),
        L_1 * np.sin(theta_f[i]),
        0.0
    ])

    # Punt F op driehoek EGF
    F[i] = E[i] + np.array([
        r_8 * np.cos(theta_8[i]),
        r_8 * np.sin(theta_8[i]),
        0.0
    ])

    # Punt G op driehoek EGF
    G[i] = E[i] + np.array([
        L_2 * np.cos(theta_8[i] - theta_I),
        L_2 * np.sin(theta_8[i] - theta_I),
        0.0
    ])

Plotten van de angulaire snelheden en versnellingen

In [ ]:
# Plot angular velocities with respect to time in subplots
plt.ion()
plt.figure()
plt.subplot(3, 3, 1)
plt.plot(t, dtheta_2, label=r'$\dot{\theta}_2$')
plt.ylabel(r' $\omega_{O2}$ [rad/s]')
plt.xlabel('t [s]')
plt.subplot(3, 3, 2)
plt.ylabel(r' $\omega_{AB}$ [rad/s]')
plt.xlabel('t [s]')
plt.plot(t, dtheta_3, label=r'$\dot{\theta}_3$')
plt.subplot(3, 3, 3)
plt.ylabel(r' $\omega_{DB}$ [rad/s]')
plt.xlabel('t [s]')
plt.plot(t, dtheta_4, label=r'$\dot{\theta}_3$')
plt.subplot(3, 3, 4)
plt.ylabel(r' $\omega_{AE}$ [rad/s]')
plt.xlabel('t [s]')
plt.plot(t, dtheta_6, label=r'$\dot{\theta}_4$')
plt.subplot(3, 3, 5)
plt.ylabel(r' $\omega_{C}$ [rad/s]')
plt.xlabel('t [s]')
plt.plot(t, dtheta_7, label=r'$\dot{\theta}_5$')
plt.subplot(3, 3, 6)
plt.ylabel(r' $\omega_{E}$ [rad/s]')
plt.xlabel('t [s]')
plt.plot(t, dtheta_8, label=r'$\dot{\theta}_6$')
plt.subplot(3, 3, 7)
plt.ylabel(r' $\omega_{DE}$ [rad/s]')
plt.xlabel('t [s]')
plt.plot(t, dtheta_f, label=r'$\dot{\theta}_7$')

plt.tight_layout()
plt.show()

In [ ]:
# plot angular accelerations with respect to time in subplots
plt.ion()
plt.figure()
plt.subplot(3, 3, 1)
plt.plot(t, ddtheta_2, label=r'$\dot{\theta}_2$')
plt.ylabel(r' $\alpha_{O2}$ [rad/s²]')
plt.xlabel('t [s]')
plt.subplot(3, 3, 2)
plt.ylabel(r' $\alpha_{AB}$ [rad/s²]')
plt.xlabel('t [s]')
plt.plot(t, ddtheta_3, label=r'$\dot{\theta}_3$')
plt.subplot(3, 3, 3)
plt.ylabel(r' $\alpha_{DB}$ [rad/s²]')
plt.xlabel('t [s]')
plt.plot(t, ddtheta_4, label=r'$\dot{\theta}_3$')
plt.subplot(3, 3, 4)
plt.ylabel(r' $\alpha_{AE}$ [rad/s²]')
plt.xlabel('t [s]')
plt.plot(t, ddtheta_6, label=r'$\dot{\theta}_4$')
plt.subplot(3, 3, 5)
plt.ylabel(r' $\alpha_{C}$ [rad/s²]')
plt.xlabel('t [s]')
plt.plot(t, ddtheta_7, label=r'$\dot{\theta}_5$')
plt.subplot(3, 3, 6)
plt.ylabel(r' $\alpha_{E}$ [rad/s²]')
plt.xlabel('t [s]')
plt.plot(t, ddtheta_8, label=r'$\dot{\theta}_6$')
plt.subplot(3, 3, 7)
plt.ylabel(r' $\alpha_{DE}$ [rad/s²]')
plt.xlabel('t [s]')
plt.plot(t, ddtheta_f, label=r'$\dot{\theta}_7$')

plt.tight_layout()
plt.show()

In [ ]:
''' 
    Calculation of the center of mass of a triangle formed by three points A, B, and C, 
    with respect to a reference point. 
    The function takes the coordinates of points A, B, and C, as well as the reference point, 
    and returns the coordinates of the center of mass of the triangle relative to the reference point.
'''

def centre_of_mass_triangle(A, B, C, reference_point): 
    # Calculate the centroid of the triangle formed by points A, B, and C
    A = A - reference_point
    B = B - reference_point
    C = C - reference_point
    x_com = (A[0] + B[0] + C[0]) / 3
    y_com = (A[1] + B[1] + C[1]) / 3
    return np.array([x_com, y_com, 0])

"""
Hier ben ik nu de versnellingen van alle massacenters aan het berekenen, 
gegeven de hoeken, hoeksnelheden en hoekversnellingen van alle links.
"""

def get_linear_velocities(r_2, r_3, L_1, r_6, r_7, r_DC, 
                          theta_2, theta_3, theta_4, theta_6, theta_7, theta_8,  theta_C, theta_f,
                          dtheta_2, dtheta_3, dtheta_4, dtheta_6, dtheta_7, dtheta_8, dtheta_f,
                          D, B, C, E, G, F):
    # snelheden van de joints
    v_A = np.cross([0,0,dtheta_2], [(r_2)*np.cos(theta_2), (r_2)*np.sin(theta_2), 0])
    v_B = np.cross([0,0,dtheta_4], [(r_4)*np.cos(theta_4), (r_4)*np.sin(theta_4), 0])
    v_C = np.cross([0,0,dtheta_4], [(r_DC)*np.cos(theta_4 + theta_C), (r_DC)*np.sin(theta_4 + theta_C), 0])
    v_E  = np.cross([0,0,dtheta_f], [(L_1)*np.cos(theta_f), (L_1)*np.sin(theta_f), 0])
    v_F = v_E + np.cross([0,0,dtheta_8], [(r_8)*np.cos(theta_8), (r_8)*np.sin(theta_8), 0])
    v_G =  v_E + np.cross([0,0,dtheta_8], [(L_2)*np.cos(theta_8 - theta_I), (L_2)*np.sin(theta_8 - theta_I), 0])

    # snelheden van de massacentra van de links
    v_1 = np.cross([0,0,dtheta_2], [(r_2 / 2)*np.cos(theta_2), (r_2/2)*np.sin(theta_2), 0]) # lineair velocity of O2A
    v_2 = v_A + np.cross([0,0,dtheta_3], [(r_3 / 2)*np.cos(theta_3), (r_3/2)*np.sin(theta_3), 0]) # velocity of com of AB 
    v_3 = v_A + np.cross([0,0,dtheta_6], [(r_6 / 2)*np.cos(theta_6), (r_6/2)*np.sin(theta_6), 0]) #velocity of com of AE
    v_4 = np.cross([0,0, dtheta_4], centre_of_mass_triangle(D,B,C,D)) # velocity of com of DBC
    v_5 = np.cross([0,0,dtheta_f], [(L_1/2)*np.cos(theta_f), (L_1/2)*np.sin(theta_f), 0]) # velocity of com of DE
    v_6 = v_C + np.cross([0,0, dtheta_7], [(r_7/2)*np.cos(theta_7), (r_7/2)*np.sin(theta_7), 0]) # velocity of com of CF
    v_7 = v_E + np.cross([0,0,dtheta_8], centre_of_mass_triangle(E,G,F,E)) # velocity of com of EGF

    return [v_1, v_2, v_3, v_4, v_5, v_6, v_7], [v_A, v_B, v_C, v_E, v_F, v_G] # bij het oproepen van deze functie moet je index [0] of [1] gebruiken om bepaalde snelheden op te roepen


def get_linear_acceleration(
        r_2, r_3, L_1, r_6, r_7, r_DC, theta_C,
        theta_2, dtheta_2, ddtheta_2,
        theta_3, dtheta_3, ddtheta_3,
        theta_4, dtheta_4, ddtheta_4, 
        theta_6, dtheta_6, ddtheta_6, 
        theta_7, dtheta_7, ddtheta_7, 
        theta_8, dtheta_8, ddtheta_8,
        theta_f, dtheta_f, ddtheta_f, 
        D, B, C, E, G, F):
    
    [v_1, v_2, v_3, v_4, v_5, v_6, v_7] = get_linear_velocities(r_2, r_3, L_1, r_6, r_7, r_DC, 
                                                                theta_2, theta_3, theta_4, theta_6, theta_7, theta_8,  theta_C, theta_f,
                                                                dtheta_2, dtheta_3, dtheta_4, dtheta_6, dtheta_7, dtheta_8, dtheta_f,
                                                                D, B, C, E, G, F)[0]
    
    # lineaire versnellingen van de joints
    a_A = np.cross([0,0,dtheta_2], np.cross([0,0,dtheta_2], [(r_2)*np.cos(theta_2), (r_2)*np.sin(theta_2), 0])) + np.cross([0,0,ddtheta_2], [(r_2)*np.cos(theta_2), (r_2)*np.sin(theta_2), 0])
    a_B = np.cross([0,0,dtheta_4], np.cross([0,0,dtheta_4], [(r_4)*np.cos(theta_4), (r_4)*np.sin(theta_4), 0])) + np.cross([0,0,ddtheta_4], [(r_4)*np.cos(theta_4), (r_4)*np.sin(theta_4), 0]) # gekeken vanuit D, check zal vanuit A zijn
    a_C = np.cross([0,0,dtheta_4], np.cross([0,0,dtheta_4], [(r_DC)*np.cos(theta_4 + theta_C), (r_DC)*np.sin(theta_4 + theta_C), 0])) + np.cross([0,0,ddtheta_4], [(r_DC)*np.cos(theta_4 + theta_C), (r_DC)*np.sin(theta_4 + theta_C), 0])
    a_E = np.cross([0,0,dtheta_f], np.cross([0,0,dtheta_f], [(L_1)*np.cos(theta_f), (L_1)*np.sin(theta_f), 0])) + np.cross([0,0,ddtheta_f], [(L_1)*np.cos(theta_f), (L_1)*np.sin(theta_f), 0])
    a_F = a_E + np.cross([0,0,dtheta_8], np.cross([0,0,dtheta_8], [(r_8)*np.cos(theta_8), (r_8)*np.sin(theta_8), 0])) + np.cross([0,0,ddtheta_8], [(r_8)*np.cos(theta_8), (r_8)*np.sin(theta_8), 0])
    a_G =  a_E + np.cross([0,0,dtheta_8], np.cross([0,0,dtheta_8], [(L_2)*np.cos(theta_8 - theta_I), (L_2)*np.sin(theta_8 - theta_I), 0])) + np.cross([0,0,ddtheta_8], [(L_2)*np.cos(theta_8 - theta_I), (L_2)*np.sin(theta_8 - theta_I), 0])

    # lineaire versnellingen van de massacentra van de links
    a_2 = np.cross([0,0,dtheta_2], v_1) + np.cross([0,0,ddtheta_2], [(r_2 / 2)*np.cos(theta_2), (r_2/2)*np.sin(theta_2), 0])
    a_3 = a_A + np.cross([0,0,dtheta_3], v_2) + np.cross([0,0,ddtheta_3], [(r_3 / 2)*np.cos(theta_3), (r_3/2)*np.sin(theta_3), 0])
    a_6 = a_A + np.cross([0,0,dtheta_6], v_3) + np.cross([0,0,ddtheta_6], [(r_6 / 2)*np.cos(theta_6), (r_6/2)*np.sin(theta_6), 0])
    a_DBC = np.cross([0,0, dtheta_4], v_4) + np.cross([0,0, ddtheta_4], centre_of_mass_triangle(D,B,C,D))
    a_L = np.cross([0,0,dtheta_f], v_5) + np.cross([0,0,ddtheta_f], [(L_1/2)*np.cos(theta_f), (L_1/2)*np.sin(theta_f), 0])
    a_7 = a_C + np.cross([0,0, dtheta_7], v_6) + np.cross([0,0, ddtheta_7], [(r_7/2)*np.cos(theta_7), (r_7/2)*np.sin(theta_7), 0])
    a_EGF = a_E + np.cross([0,0,dtheta_8], v_7) + np.cross([0,0,ddtheta_8], centre_of_mass_triangle(E,G,F,E))

    return [a_2, a_3, a_6, a_DBC, a_L, a_7, a_EGF], [a_A, a_B, a_C, a_E, a_F, a_G]

"""
Nu maken we arrays aan om de lineaire snelheden en versnellingen van alle massacenters op te slaan
"""
#snelheden van de joints
vA = np.zeros((len(theta_2), 3))
vB = np.zeros((len(theta_2), 3))
vC = np.zeros((len(theta_2), 3))
vE = np.zeros((len(theta_2), 3))
vF = np.zeros((len(theta_2), 3))
vG = np.zeros((len(theta_2), 3))

#snelheden van de massacentra van de links
v1 = np.zeros((len(theta_2), 3))
v2 = np.zeros((len(theta_2), 3))
v3 = np.zeros((len(theta_2), 3))
v4 = np.zeros((len(theta_2), 3))
v5 = np.zeros((len(theta_2), 3))
v6 = np.zeros((len(theta_2), 3))
v7 = np.zeros((len(theta_2), 3))

#versnellingen van de joints
aA = np.zeros((len(theta_2), 3))
aB = np.zeros((len(theta_2), 3))
aC = np.zeros((len(theta_2), 3))
aE = np.zeros((len(theta_2), 3))
aF = np.zeros((len(theta_2), 3))
aG = np.zeros((len(theta_2), 3)) 

# versnellingen van de massacentra van de links
a2 = np.zeros((len(theta_2), 3))
a3 = np.zeros((len(theta_2), 3))
a6 = np.zeros((len(theta_2), 3))
aDBC = np.zeros((len(theta_2), 3))
aL = np.zeros((len(theta_2), 3))
a7 = np.zeros((len(theta_2), 3))
aEGF = np.zeros((len(theta_2), 3))

for i in range(len(theta_2)):
    v_points = get_linear_velocities(
        r_2, r_3, L_1, r_6, r_7, r_DC,
        theta_2[i], theta_3[i], theta_4[i], theta_6[i], theta_7[i], theta_8[i], theta_C, theta_f[i],
        dtheta_2[i], dtheta_3[i], dtheta_4[i], dtheta_6[i], dtheta_7[i], dtheta_8[i], dtheta_f[i],
        D[i], B[i], C[i], E[i], G[i], F[i]
    )[1]

    v_coms = get_linear_velocities(
        r_2, r_3, L_1, r_6, r_7, r_DC,
        theta_2[i], theta_3[i], theta_4[i], theta_6[i], theta_7[i], theta_8[i], theta_C, theta_f[i],
        dtheta_2[i], dtheta_3[i], dtheta_4[i], dtheta_6[i], dtheta_7[i], dtheta_8[i], dtheta_f[i],
        D[i], B[i], C[i], E[i], G[i], F[i]
    )[0]

    aA[i], aB[i], aC[i], aE[i], aF[i], aG[i] = get_linear_acceleration(
        r_2, r_3, L_1, r_6, r_7, r_DC, theta_C,
        theta_2[i], dtheta_2[i], ddtheta_2[i],
        theta_3[i], dtheta_3[i], ddtheta_3[i],
        theta_4[i], dtheta_4[i], ddtheta_4[i],
        theta_6[i], dtheta_6[i], ddtheta_6[i],
        theta_7[i], dtheta_7[i], ddtheta_7[i],
        theta_8[i], dtheta_8[i], ddtheta_8[i],
        theta_f[i], dtheta_f[i], ddtheta_f[i],
        D[i], B[i], C[i], E[i], G[i], F[i]
    )[1]

    a2[i], a3[i], a6[i], aDBC[i], aL[i], a7[i], aEGF[i] = get_linear_acceleration(
        r_2, r_3, L_1, r_6, r_7, r_DC, theta_C,
        theta_2[i], dtheta_2[i], ddtheta_2[i],
        theta_3[i], dtheta_3[i], ddtheta_3[i],
        theta_4[i], dtheta_4[i], ddtheta_4[i],
        theta_6[i], dtheta_6[i], ddtheta_6[i],
        theta_7[i], dtheta_7[i], ddtheta_7[i],
        theta_8[i], dtheta_8[i], ddtheta_8[i],
        theta_f[i], dtheta_f[i], ddtheta_f[i],
        D[i], B[i], C[i], E[i], G[i], F[i]
    )[0]

    v1[i], v2[i], v3[i], v4[i], v5[i], v6[i], v7[i] = v_coms
    vA[i], vB[i], vC[i], vE[i], vF[i], vG[i] = v_points

"""
X en Y componenten van de lineaire snelheden en versnellingen van alle massacenters selecteren
"""
v1_x = v1[:,1] 
v1_y = v1[:,0]

a2_x = a2[:, 0]
a2_y = a2[:, 1]

a3_x = a3[:, 0]
a3_y = a3[:, 1]

a6_x = a6[:, 0]
a6_y = a6[:, 1]

aDBC_x = aDBC[:, 0]
aDBC_y = aDBC[:, 1]

aL_x = aL[:, 0]
aL_y = aL[:, 1]

a7_x = a7[:, 0]
a7_y = a7[:, 1]

aEGF_x = aEGF[:, 0]
aEGF_y = aEGF[:, 1]

Check van snelheden en versnellingen

In [ ]:
#check snelheden van de joints
vA_check = np.zeros((len(theta_2), 3))
vB_check = np.zeros((len(theta_2), 3))
vC_check = np.zeros((len(theta_2), 3))
vE_check = np.zeros((len(theta_2), 3))
vF_check = np.zeros((len(theta_2), 3))


def check_linear_velocities(r_2, r_3, L_1, r_6, r_7, r_DC, 
                          theta_2, theta_3, theta_4, theta_6, theta_7, theta_8,  theta_C, theta_f,
                          dtheta_2, dtheta_3, dtheta_4, dtheta_6, dtheta_7, dtheta_8, dtheta_f,
                          D, B, C, E, G, F):
    # snelheden van de joints
    vA_check = np.cross([0,0,dtheta_2], [(r_2)*np.cos(theta_2), (r_2)*np.sin(theta_2), 0])
    vB_check = vA_check + np.cross([0,0,dtheta_3], [(r_3)*np.cos(theta_3), (r_3)*np.sin(theta_3), 0])
    vE_check = vA_check + np.cross([0,0,dtheta_6], [(r_6)*np.cos(theta_6), (r_6)*np.sin(theta_6), 0])
    vC_check = np.cross([0,0,dtheta_4], [(r_DC)*np.cos(theta_4 + theta_C), (r_DC)*np.sin(theta_4 + theta_C), 0])
    vF_check = vC_check + np.cross([0,0,dtheta_7], [(r_7)*np.cos(theta_7), (r_7)*np.sin(theta_7), 0])

    return [vA_check, vB_check, vE_check, vC_check, vF_check]

for i in range(len(theta_2)):
    vA_check[i], vB_check[i], vE_check[i], vC_check[i], vF_check[i] = check_linear_velocities(r_2, r_3, L_1, r_6, r_7, r_DC,
                    theta_2[i], theta_3[i], theta_4[i], theta_6[i], theta_7[i], theta_8[i], theta_C, theta_f[i],
                    dtheta_2[i], dtheta_3[i], dtheta_4[i], dtheta_6[i], dtheta_7[i], dtheta_8[i], dtheta_f[i],
                    D[i], B[i], C[i], E[i], G[i], F[i])
plt.ion()
plt.figure()


#v_B checks dtheta_3 and dtheta_4
plt.subplot(3, 3, 1)
plt.plot(t, vB_check - vB)
plt.xlabel('t [s]')
plt.ylabel(r'$v_B$ [cm/s]')
plt.legend(["x", "y"])

#v_E checks dtheta_f and dtheta_6
plt.subplot(3, 3, 2)
plt.plot(t, vE_check - vE)
plt.xlabel('t [s]')
plt.ylabel(r'$v_E$ [cm/s]')
plt.legend(["x", "y"])

#v_F checks dtheta_8 and dtheta_7
plt.subplot(3, 3, 3)
plt.plot(t, vF_check - vF)
plt.xlabel('t [s]')
plt.ylabel(r'$v_F$ [cm/s]')
plt.legend(["x", "y"])

        
plt.tight_layout()


In [ ]:
aA_check = np.zeros((len(theta_2), 3))
aB_check = np.zeros((len(theta_2), 3))
aC_check = np.zeros((len(theta_2), 3))
aE_check = np.zeros((len(theta_2), 3))
aF_check = np.zeros((len(theta_2), 3))

def check_linear_accelerations(r_2, r_3, L_1, r_6, r_7, r_DC, 
                          theta_2, theta_3, theta_4, theta_6, theta_7, theta_8,  theta_C, theta_f,
                          dtheta_2, dtheta_3, dtheta_4, dtheta_6, dtheta_7, dtheta_8, dtheta_f,
                          ddtheta_2, ddtheta_3, ddtheta_4, ddtheta_6, ddtheta_7, ddtheta_8, ddtheta_f,
                          D, B, C, E, G, F):
    # versnellingen van de joints
    
    aA_check = np.cross([0,0,dtheta_2], np.cross([0,0,dtheta_2], [(r_2)*np.cos(theta_2), (r_2)*np.sin(theta_2), 0])) + np.cross([0,0,ddtheta_2], [(r_2)*np.cos(theta_2), (r_2)*np.sin(theta_2), 0])
    aB_check = aA_check + np.cross([0,0,dtheta_3], np.cross([0,0,dtheta_3], [(r_3)*np.cos(theta_3), (r_3)*np.sin(theta_3), 0])) + np.cross([0,0,ddtheta_3], [(r_3)*np.cos(theta_3), (r_3)*np.sin(theta_3), 0])
    aE_check = aA_check + np.cross([0,0,dtheta_6], np.cross([0,0,dtheta_6], [(r_6)*np.cos(theta_6), (r_6)*np.sin(theta_6), 0])) + np.cross([0,0,ddtheta_6], [(r_6)*np.cos(theta_6), (r_6)*np.sin(theta_6), 0])
    aC_check = np.cross([0,0,dtheta_4], np.cross([0,0,dtheta_4], [(r_DC)*np.cos(theta_4 + theta_C), (r_DC)*np.sin(theta_4 + theta_C), 0])) + np.cross([0,0,ddtheta_4], [(r_DC)*np.cos(theta_4 + theta_C), (r_DC)*np.sin(theta_4 + theta_C), 0])
    aF_check = aC_check + np.cross([0,0,dtheta_7], np.cross([0,0,dtheta_7], [(r_7)*np.cos(theta_7), (r_7)*np.sin(theta_7), 0])) + np.cross([0,0,ddtheta_7], [(r_7)*np.cos(theta_7), (r_7)*np.sin(theta_7), 0])

    return [aA_check, aB_check, aC_check, aE_check, aF_check]

for i in range(len(theta_2)):
    aA_check[i], aB_check[i], aC_check[i], aE_check[i], aF_check[i] = check_linear_accelerations(r_2, r_3, L_1, r_6, r_7, r_DC,
                    theta_2[i], theta_3[i], theta_4[i], theta_6[i], theta_7[i], theta_8[i], theta_C, theta_f[i],
                    dtheta_2[i], dtheta_3[i], dtheta_4[i], dtheta_6[i], dtheta_7[i], dtheta_8[i], dtheta_f[i],
                    ddtheta_2[i], ddtheta_3[i], ddtheta_4[i], ddtheta_6[i], ddtheta_7[i], ddtheta_8[i], ddtheta_f[i],
                    D[i], B[i], C[i], E[i], G[i], F[i])
    
plt.ion()
plt.figure()


#v_B checks dtheta_3 and dtheta_4
plt.subplot(3, 3, 1)
plt.plot(t, aB_check - aB)
plt.xlabel('t [s]')
plt.ylabel(r'$a_B$ [cm/s²]')
plt.legend(["x", "y"])

#v_E checks dtheta_f and dtheta_6
plt.subplot(3, 3, 2)
plt.plot(t, aE_check - aE)
plt.xlabel('t [s]')
plt.ylabel(r'$a_E$ [cm/s²]')
plt.legend(["x", "y"])

#v_F checks dtheta_8 and dtheta_7
plt.subplot(3, 3, 3)
plt.plot(t, aF_check - aF)
plt.xlabel('t [s]')
plt.ylabel(r'$a_F$ [cm/s²]')
plt.legend(["x", "y"])

        
plt.tight_layout()


Check obv centrale differentie

In [ ]:
for point in ['A', 'B','C', 'E', 'F', 'G']:
    vP_disc = np.zeros((len(theta_2), 3))
    aP_disc = np.zeros((len(theta_2), 3))  
    rP, vP, aP = globals()[point], globals()[f'v{point}'], globals()[f'a{point}']

    for i in range(2,len(theta_2)-2):
        #vP_disc[i] = (rP[i+1] - rP[i-1]) / (t[i+1]-t[i-1])
        #aP_disc[i] = (vP[i+1] - vP[i-1]) / (t[i+1]-t[i-1])
        vP_disc[i] = (-rP[i+2] + 8*rP[i+1] - 8*rP[i-1] + rP[i-2]) / (12*(t[i+1]-t[i])) # min moeten toevoegen omdat we in de negatieve zin draaien
        aP_disc[i] = (-vP[i+2] + 8*vP[i+1] - 8*vP[i-1] + vP[i-2]) / (12*(t[i+1]-t[i]))
    
    vP_disc[0] = vP[0]  
    aP_disc[0] = aP[0]
    vP_disc[1] = vP[1]  
    aP_disc[1] = aP[1]
    vP_disc[len(theta_2)-1] = vP[len(theta_2)-1]  
    aP_disc[len(theta_2)-1] = aP[len(theta_2)-1]
    vP_disc[len(theta_2)-2] = vP[len(theta_2)-2]  
    aP_disc[len(theta_2)-2] = aP[len(theta_2)-2]

    plt.figure()
    plt.subplot(2,2,1)
    plt.plot(t, vP, label=rf'$v_{point}$')
    plt.xlabel('t [s]')
    plt.ylabel(rf'$Analytical\ v_{point}$ [cm/s]')
    plt.legend(["x", "y"])
    plt.subplot(2,2,2)
    plt.plot(t, vP - vP_disc, label=rf'$v_{point}$')
    plt.xlabel('t [s]')
    plt.ylabel(rf'$Absolute\ error\ v_{point}$ [cm/s]')
    plt.legend(["x", "y"])

    plt.subplot(2,2,3)
    plt.plot(t, aP, label=rf'$a_{point}$')
    plt.xlabel('t [s]')
    plt.ylabel(rf'$Analytical\ a_{point}$ [cm/s²]')
    plt.legend(["x", "y"])
    plt.subplot(2,2,4)
    plt.plot(t, aP - aP_disc, label=rf'$a_{point}$')
    plt.xlabel('t [s]')
    plt.ylabel(rf'$Absolute\ error\ a_{point}$ [cm/s²]')
    plt.legend(["x", "y"])

    plt.tight_layout()
    plt.show()

# Inverse Dynamica
We hebben nu de beweging van het mechanisme bepaald. In het deel *inverse dynamica* zijn we geïnteresseerd in de reactiekrachten in de scharnieren en het aandrijfkoppel dat nodig is voor de beweging. Hiervoor hebben we de volledige *beweging* (positie, snelheid, versnelling) en de *externe krachten* als gegevens nodig.

Hiermee genereren we voor elk van de staven 3 evenwichts-vergelijkingen:

$\sum F_x = ma_x$,

$\sum F_y = ma_y$,

$\sum M_z + \sum \vec{r} \times \vec{F} = I\alpha$.

Voordat we hieraan beginnen definiëren we hieronder eerst een aantal hulpfuncties

In [ ]:
'''
    Calculation of the center of mass of a link, given its length and angle. 
    Assuming uniform density, the center of mass is at the midpoint of the link, 
    with respect to an origin at the beginning of the link.
'''

def centre_of_mass_link(length_link, angle_link): # niet gebruikt
    # Assuming uniform density, the center of mass is at the midpoint of the link
    x_com = (length_link / 2) * np.cos(angle_link)
    y_com = (length_link / 2) * np.sin(angle_link)
    return np.array([x_com, y_com])

''' 
    Calculation of the center of mass of a triangle formed by three points A, B, and C, 
    with respect to a reference point. 
    The function takes the coordinates of points A, B, and C, as well as the reference point, 
    and returns the coordinates of the center of mass of the triangle relative to the reference point.
'''

def centre_of_mass_triangle(A, B, C, reference_point): 
    # Calculate the centroid of the triangle formed by points A, B, and C
    A = A - reference_point
    B = B - reference_point
    C = C - reference_point
    x_com = (A[0] + B[0] + C[0]) / 3
    y_com = (A[1] + B[1] + C[1]) / 3
    return np.array([x_com, y_com, 0])


'''
    Calculation of the moment of inertia of a link about its center of mass, given its mass and length. 
    Assuming uniform density, the moment of inertia is given by the formula I = (1/12) * mass * length^2.
'''

def moment_of_inertia_com_link(mass, length_link):
    I_com = (1/12) * mass * length_link**2
    return I_com

'''
    Calculation of the moment of inertia of a link about an axis through a reference point, 
    given its mass and length. 
'''

def moment_of_inertia_link(mass, length_link, reference_point): # niet gebruikt
    # Assuming uniform density, the moment of inertia of a link about its center of mass is given by:
    I_com = moment_of_inertia_com_link(mass, length_link)
    com = reference_point  # Placeholder - replace with actual center of mass calculation
    I = I_com + mass * np.linalg.norm(com - reference_point)**2
    return I

'''
    Calculation of the moment of inertia of a triangle formed by three points A, B, and C, 
    with respect to its center of mass. 
    The function takes the coordinates of points A, B, and C, as well as the density of the material, 
    and returns the moment of inertia of the triangle about an axis through its center of mass.
'''

def moment_of_inertia_com_triangle(A, B, C, density):
    area = 0.5*np.abs(A[0]*(B[1]-C[1]) + B[0]*(C[1]-A[1]) + C[0]*(A[1]-B[1]))
    mass = area * density  # Assuming uniform density, mass is proportional to area

    a = np.norm(A)
    b = np.norm(B)
    c = np.norm(C)

    I_com = mass * (a**2 + b**2 + c**2) / 36
    return I_com

'''
    Calculation of the moment of inertia of a triangle formed by three points A, B, and C, 
    about an axis through a reference point. 
    The function takes the coordinates of points A, B, and C, as well as the reference point, 
    and returns the moment of inertia of the triangle about an axis through the reference point.
'''

def moment_of_inertia_triangle(A, B, C, reference_point): # niet gebruikt
    com = centre_of_mass_triangle(A, B, C, reference_point)
    I_com = moment_of_inertia_com_triangle(A, B, C, density=1)

    I = I_com + mass * np.linalg.norm(com - reference_point)**2
    return I

"""
Dit geeft de coordinaten van E en F t.o.v. het massacentrum van de driehoek EFG, 
gegeven de lengte van EG, EF, en de hoeken GEF en EF t.o.v. de positieve horizontale as.
"""

def coordinaten_tov_massacentrum_driehoek(L_2, r_8, theta_I, theta_8):
    # Absolute coördinaten nemen met E als oorsprong
    E_abs_x = 0.0
    E_abs_y = 0.0
    # Punt F: hoek theta_8 met de klok mee => y negatief bij positieve theta_8
    F_abs_x = r_8 * np.cos(theta_8)
    F_abs_y = r_8 * np.sin(theta_8)
    # Punt G: theta_I verder met de klok mee zodat G "naar beneden" ligt
    G_abs_x = L_2 * np.cos(theta_8 - theta_I)
    G_abs_y = L_2 * np.sin(theta_8 - theta_I)
    # Massacentrum van homogene driehoek
    x_c = (E_abs_x + F_abs_x + G_abs_x) / 3
    y_c = (E_abs_y + F_abs_y + G_abs_y) / 3
    # Coördinaten t.o.v. massacentrum
    E_x = E_abs_x - x_c
    E_y = E_abs_y - y_c
    F_x = F_abs_x - x_c
    F_y = F_abs_y - y_c
    G_x = G_abs_x - x_c
    G_y = G_abs_y - y_c

    return E_x, E_y, F_x, F_y, G_x, G_y

def moment_of_inertia_massacentrum(a, b, theta, rho): #voor een driehoek
    A = 0.5 * a * b * np.sin(theta)     # Oppervlakte
    m = rho * A    # Massa 
    J_O = rho * A * (a**2 + b**2) / 6    # Traagheidsmoment rond hoekpunt
    d2 = (a**2 + b**2 + 2*a*b*np.cos(theta)) / 9 # Afstand zwaartepunt tot hoekpunt
    J_G = J_O - m * d2    # Steiner
    return J_G

def moment_of_inertia_hoekpunt(a, b, theta, rho):
    A = 0.5 * a * b * np.sin(theta) # Oppervlakte
    J = rho * A * (a**2 + b**2) / 6
    return J

In [ ]:
"""
Eerst bepaal ik de massas van alle staven en driehoeeken, op basis van een gegeven dichtheid en de lengte van de staven
en de oppervlakte van de driehoeken. De dikte van de staven wordt en driehoeken wordt momenteel cte. 1cm verondersteld. 
De staven een breedte van 4cm.
"""
rho_staff = 1600 * 0.01 * 0.04 # massa van 60% gevulde aluminium in kg/m^3 (dikte 1cm, breedte 4cm)
rho_triangle = 800 * 0.01 # massa van 30% gevulde aluminium in kg/m^3 (dikte 1cm)

"""
Voor het gewicht van de dij staaf wordt het gewicht van de persoon vermenigvuldigd
met het percentage van het lichaamsgewicht dat op de dij zit. Zo brengen we de inertie
van de dij in rekening zonder dat we een gecompliceerd model van het been nodig hebben.
"""
m_persoon = 70 # mass of the person in kg
m_dij = 0.1416 * m_persoon # uit paper gehaald

m2 = r_2 * rho_staff # mass of link O2A
m3 = r_3 * rho_staff # mass of link AB
m6 = r_6 * rho_staff # mass of link AE
mDBC = 0.5 * r_4 * r_DC * np.sin(theta_C) * rho_triangle # mass of triangle DBC
mL = L_1 * 0.01 * rho_staff + m_dij # mass of link DE
m7 = r_7 * rho_staff # mass of link CF
mEGF = 0.5 * L_2 * r_8 * np.sin(theta_I) * rho_triangle # mass of triangle EGF


In [ ]:
"""
Nu bepaal ik alle traagheidsmomenten.
- voor staven R2 en L1 ten opzichte van O en D respectievelijk
- voor staven R3, R6 en R7 ten opzichte van hun eigen middelpunt
- voor driehoek DBC ten opzichte van D
- voor driehoek EGF ten opzichte van zijn eigen middelpunt
"""
# Polaire traagheidsmomenten van R2 en L1 (zwaartepunt rechthoekige balk + stelling van parallelle assen)
J2 = (1/12) * m2 * r_2**2 + m2 * (r_2/2)**2
J_L = (1/12) * mL * L_1**2 + mL * (L_1/2)**2

# Polaire traagheidsmomenten van R3, R6 en R7 ten opzichte van hun eigen middelpunt
J3 = (1/12) * m3 * r_3**2
J6 = (1/12) * m6 * r_6**2
J7 = (1/12) * m7 * r_7**2

# Traagheidsmoment van driehoek DBC ten opzichte van D
J_DBC = moment_of_inertia_hoekpunt(r_4, r_DC, theta_C, rho_triangle)

# Traagheidsmoment van driehoek EGF ten opzichte van zijn eigen middelpunt
J_EGF = moment_of_inertia_massacentrum(r_8, L_2, theta_I, rho_triangle)

In [ ]:
"""
functie om x-afstanden te bepalen van zwaartepunt EGF en G en F.
"""

### Oplossing Inverse dynamica
#### 27 vergelijkingen en 27 onbekenden, zoals hieronder uitgelegd:
**Vergelijkingen:**
- 7 staven geven elk 3 vergelijkingen: 21 vgl
- In meervoudige knopen A, D & E moeten staafkrachten mekaar opheffen (in x en y richting): 3 * 2 = 6 vgl
- totaal: 27 vgl

**Onbekenden**
- knopen A, D en E geven elk 6 onbekenden: 18 onb.
- knopen D, C, O & F geven elk 2 onbekenden: 8 onb.
- aandrijfkoppel in O: 1 onb.
- totaal: 27 onb.

In [ ]:
g = 0 #momenteel nog geen zwaartekracht (als je wel toevoegd moet er een minteken voor)

# Matrices voor krachten en momenten
F_O_x = np.zeros_like(theta_2)
F_O_y = np.zeros_like(theta_2)
F_A2_x = np.zeros_like(theta_2)
F_A2_y = np.zeros_like(theta_2)
F_A3_x = np.zeros_like(theta_2)
F_A3_y = np.zeros_like(theta_2)
F_A6_x = np.zeros_like(theta_2)
F_A6_y = np.zeros_like(theta_2)
F_B_x = np.zeros_like(theta_2)
F_B_y = np.zeros_like(theta_2)
F_E6_x = np.zeros_like(theta_2)
F_E6_y = np.zeros_like(theta_2)
F_D_x = np.zeros_like(theta_2)
F_D_y = np.zeros_like(theta_2)
F_DL_x = np.zeros_like(theta_2)
F_DL_y = np.zeros_like(theta_2)
F_EL_x = np.zeros_like(theta_2)
F_EL_y = np.zeros_like(theta_2)
F_C_x = np.zeros_like(theta_2)
F_C_y = np.zeros_like(theta_2)
F_F_x = np.zeros_like(theta_2)
F_F_y = np.zeros_like(theta_2)
F_DT_x = np.zeros_like(theta_2)
F_DT_y = np.zeros_like(theta_2)
F_ET_x = np.zeros_like(theta_2)
F_ET_y = np.zeros_like(theta_2)
M_0 = np.zeros_like(theta_2)

# Oplossen van kracht en moment evenwichten voor elke tijdstap
for i in range(len(t)):
        E_x, E_y, F_x, F_y, G_x, G_y = coordinaten_tov_massacentrum_driehoek(L_2, r_8, theta_I, theta_8[i])
#FO_x, FO_y, FA2_x, FA2_y, FA3_x, FA3_y, FA6_x, FA6_y, FB_x, FB_y, FE6_x, FE6_y, FD_x, FD_y, FDL_x, FDL_y, FEL_x, FEL_y, FC_x, FC_y, FF_x, FF_y, FDT_x, FDT_y, FET_x, FET_y, M_0 = np.linalg.solve(A, b)
        AF = np.array([[1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],     #staaf R2
                      [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                      [0, 0, -r_2 * np.sin(theta_2[i]), r_2 * np.cos(theta_2[i]), 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
                      
                      [0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],    #staaf R3
                      [0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                      [0, 0, 0, 0, 0.5 * r_3 * np.sin(theta_3[i]), -0.5 * r_3 * np.cos(theta_3[i]), 0, 0, -0.5 * r_3 * np.sin(theta_3[i]), 0.5 * r_3 * np.cos(theta_3[i]), 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                      
                      [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],    #staaf R6
                      [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                      [0, 0, 0, 0, 0, 0, 0.5 * r_6 * np.sin(theta_6[i]), -0.5 * r_6 * np.cos(theta_6[i]), 0, 0, -0.5 * r_6 * np.sin(theta_6[i]), 0.5 * r_6 * np.cos(theta_6[i]), 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],

                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],    #staaf L1
                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -L_1 * np.sin(theta_f[i]), L_1 * np.cos(theta_f[i]), 0, 0, 0, 0, 0, 0, 0, 0, 0],

                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0],    #staaf R7
                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0],
                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.5 * r_7 * np.sin(theta_7[i]), -0.5 * r_7 * np.cos(theta_7[i]), -0.5 * r_7 * np.sin(theta_7[i]), 0.5 * r_7 * np.cos(theta_7[i]), 0, 0, 0, 0, 0],
                      
                      [0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 1, 0, 0, 0, 0],    #driehoek DBC
                      [0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 1, 0, 0, 0],
                      [0, 0, 0, 0, 0, 0, 0, 0, r_4 * np.sin(theta_4[i]), -r_4 * np.cos(theta_4[i]), 0, 0, 0, 0, 0, 0, 0, 0, r_DC * np.sin(theta_4[i]+theta_C), -r_DC * np.cos(theta_4[i]+theta_C), 0, 0, 0, 0, 0, 0, 0],

                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 1, 0, 0],    #driehoek EGF
                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 1, 0],
                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, F_y, -F_x, 0, 0, -E_y, E_x, 0],
                
                      [0, 0, -1, 0, -1, 0, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #knoop A
                      [0, 0, 0, -1, 0, -1, 0, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],      

                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, -1, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 0], #knoop D
                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, -1, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0],

                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 0, 0, -1, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0], #knoop E
                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 0, 0, -1, 0, 0, 0, 0, 0, 0, 0, -1, 0]             
                      ])

        BF = np.array([m2 * a2_x[i],
                      m2 * (a2_y[i]+g),
                      J2 * ddtheta_2[i],

                      m3 * a3_x[i],
                      m3 * (a3_y[i]+g),
                      J3 * ddtheta_3[i],

                      m6 * a6_x[i],
                      m6 * (a6_y[i]+g),
                      J6 * ddtheta_6[i],

                      mL * aL_x[i],
                      mL * (aL_y[i]+g),
                      J_L * ddtheta_f[i],

                      m7 * a7_x[i],
                      m7 * (a7_y[i]+g),
                      J7 * ddtheta_7[i],

                      mDBC * aDBC_x[i],
                      mDBC * (aDBC_y[i]+g),
                      J_DBC * ddtheta_4[i],

                      mEGF * aEGF_x[i],
                      mEGF * (aEGF_y[i]+g),
                      J_EGF * ddtheta_8[i], 
                                    
                      0,
                      0,
                      
                      0,
                      0,
                      
                      0,
                      0])

        x = np.linalg.lstsq(AF, BF, rcond=None)[0]

        # Save results
        F_O_x[i] = x[0]
        F_O_y[i] =  x[1]
        F_A2_x[i] = x[2]
        F_A2_y[i] = x[3]
        F_A3_x[i] = x[4]
        F_A3_y[i] = x[5]
        F_A6_x[i] = x[6]
        F_A6_y[i] = x[7]
        F_B_x[i] = x[8]
        F_B_y[i] = x[9]
        F_E6_x[i] = x[10]
        F_E6_y[i] = x[11]
        F_D_x[i] = x[12]
        F_D_y[i] = x[13]
        F_DL_x[i] = x[14]
        F_DL_y[i] = x[15]
        F_EL_x[i] = x[16]
        F_EL_y[i] = x[17]
        F_C_x[i] = x[18]
        F_C_y[i] = x[19]
        F_F_x[i] = x[20]
        F_F_y[i] = x[21]
        F_DT_x[i] = x[22]
        F_DT_y[i] = x[23]
        F_ET_x[i] = x[24]
        F_ET_y[i] = x[25]
        M_0[i] = x[26]

In [ ]:
plt.figure()
plt.subplot(221)
plt.plot(t, F_O_x)
plt.plot(t, F_O_y)
plt.xlabel('t [s]')
plt.ylabel(r'$F_O$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.subplot(222)
plt.plot(t, F_A2_x)
plt.plot(t, F_A2_y)
plt.xlabel('t [s]')
plt.ylabel(r'$F_{A2}$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.subplot(223)
plt.plot(t, F_A3_x)
plt.plot(t, F_A3_y)
plt.xlabel('t [s]')
plt.ylabel(r'$F_{A3}$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.subplot(224)
plt.plot(t, F_A6_x)
plt.plot(t, F_A6_y)
plt.xlabel('t [s]')
plt.ylabel(r'$F_{A6}$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.tight_layout()
plt.show()


plt.subplot(221)
plt.plot(t, F_B_x)
plt.plot(t, F_B_y)
plt.xlabel('t [s]')
plt.ylabel(r'$F_B$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.subplot(222)
plt.plot(t, F_E6_x)
plt.plot(t, F_E6_y)
plt.xlabel('t [s]')
plt.ylabel(r'$F_{E6}$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.subplot(223)
plt.plot(t, F_D_x)
plt.plot(t, F_D_y)
plt.xlabel('t [s]')
plt.ylabel(r'$F_{D}$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.subplot(224)
plt.plot(t, F_DL_x)
plt.plot(t, F_DL_y)
plt.xlabel('t [s]')
plt.ylabel(r'$F_{DL}$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.tight_layout()
plt.show()

plt.subplot(221)
plt.plot(t, F_EL_x)
plt.plot(t, F_EL_y)
plt.xlabel('t [s]')
plt.ylabel(r'$F_{EL}$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.subplot(222)
plt.plot(t, F_C_x)
plt.plot(t, F_C_y)
plt.xlabel('t [s]')
plt.ylabel(r'$F_{C}$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.subplot(223)
plt.plot(t, F_F_x)
plt.plot(t, F_F_y)
plt.xlabel('t [s]')
plt.ylabel(r'$F_{F}$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.subplot(224)
plt.plot(t, F_DT_x)
plt.plot(t, F_DT_y)
plt.xlabel('t [s]')
plt.ylabel(r'$F_{DT}$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.tight_layout()
plt.show()


plt.subplot(221)
plt.plot(t, F_ET_x)
plt.plot(t, F_ET_y)
plt.xlabel('t [s]')
plt.ylabel(r'$F_{ET}$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.tight_layout()
plt.show()



In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(t, M_0)
plt.title("Aandrijfmoment M_0")
plt.xlabel("t")
plt.ylabel("M_0")
plt.grid()
plt.show()

In [ ]:
plt.figure()
plt.plot(t, F_O_x, label="F_O_x")
plt.plot(t, F_O_y, label="F_O_y")
plt.legend()
plt.title("Reactiekracht in O")
plt.grid()
plt.show()

In [ ]:
F_O = np.sqrt(F_O_x**2 + F_O_y**2) # onbalanskrachten aan de motor

plt.figure()
plt.plot(t, F_O)
plt.title("Resultante kracht in O")
plt.grid()
plt.show()

Check met variatie van kinetische energie: 

<h4>Variatie in de kinetische energie</h4>

In het geval van een zuiver inerti&euml;el systeem (d.w.z., met verwaarlozing van zwaartekracht, veren, en wrijving), is de arbeid geleverd door de aandrijvende krachten
de oorzaak van een <em>verandering</em> in de totale kinetische energie $E$.
In het geval van een kruk-drijfstang-mechanisme: 
$$\frac{dE}{dt} = M_{12} \dot{\theta}.$$

De kinetische energie is:

$$E = \frac{1}{2} \sum_{i=2}^{4} \left[
 m_i  (v_{ix}^2 + v_{iy}^2) + I_{cgi} \omega_i^2
\right],$$

$$ \text{dus} \qquad
\sum_{i=2}^{4} m_i  (v_{ix}  a_{ix} + v_{iy}  a_{iy})
  + I_{cgi}  \omega_i  \alpha_i
  =  M_{12} \dot{\theta}.
$$

Daarbij is $\dot{\theta}$ <em>gegeven</em>, en berekenen we
$v_{ix}, a_{ix}, v_{iy}, a_{iy}, \omega_i$ en $\alpha_i$
uit de kinematische analyse (&ldquo;sluitingsvergelijkingen&rdquo;).
Daarna berekenen we dan $M_{12}$.


In [ ]:
M_ext = np.zeros_like(theta_2)
'''v1[i], v2[i], v3[i], v4[i], v5[i], v6[i], v7[i] = get_linear_velocities(
        r_2, r_3, L_1, r_6, r_7, r_DC,
        theta_2[i], theta_3[i], theta_4[i], theta_6[i], theta_7[i], theta_8[i], theta_C, theta_f[i],
        dtheta_2[i], dtheta_3[i], dtheta_4[i], dtheta_6[i], dtheta_7[i], dtheta_8[i], dtheta_f[i],
        D[i], B[i], C[i], E[i], G[i], F[i]
    )[0]

a2[i], a3[i], a6[i], aDBC[i], aL[i], a7[i], aEGF[i] = get_linear_acceleration(
        r_2, r_3, L_1, r_6, r_7, r_DC, theta_C,
        theta_2[i], dtheta_2[i], ddtheta_2[i],
        theta_3[i], dtheta_3[i], ddtheta_3[i],
        theta_4[i], dtheta_4[i], ddtheta_4[i],
        theta_6[i], dtheta_6[i], ddtheta_6[i],
        theta_7[i], dtheta_7[i], ddtheta_7[i],
        theta_8[i], dtheta_8[i], ddtheta_8[i],
        theta_f[i], dtheta_f[i], ddtheta_f[i],
        D[i], B[i], C[i], E[i], G[i], F[i]
    )[0]'''

for i in range(len(t)):
    M_ext[i] = (m2*np.dot(v1[i], a2[i]) + m3*np.dot(v2[i], a3[i]) + m6*np.dot(v3[i], a6[i]) + mDBC*np.dot(v4[i], aDBC[i]) + mL*np.dot(v5[i], aL[i]) + m7*np.dot(v6[i], a7[i]) + mEGF*np.dot(v7[i], aEGF[i]) +
                (1/12) * m2 *r_2 **2 *dtheta_2[i]*ddtheta_2[i] + J3*dtheta_3[i]*ddtheta_3[i] + J6*dtheta_6[i]*ddtheta_6[i] + moment_of_inertia_massacentrum(r_4, r_DC, theta_C, rho_triangle)*dtheta_4[i]*ddtheta_4[i] +(1/12) * mL * L_1**2 * dtheta_f[i]*ddtheta_f[i] + J7*dtheta_7[i]*ddtheta_7[i] + J_EGF*dtheta_8[i]*ddtheta_8[i]) / dtheta_2[i]
    

plt.plot(t, M_ext)
plt.xlabel('t [s]')
plt.ylabel('$M_{ext}$ [Nm]')
plt.grid()
plt.title('Variation of kinetic energy')
plt.show()

plt.plot(t, M_ext - M_0)
plt.xlabel('t [s]')
plt.ylabel('Absolute error [W]')
plt.grid()
plt.title('check of external moment')
plt.show()

Berekening van de reactiekrachten obv virtuele arbeid:

Bijvoorbeeld: berekening van $F_{12x}$:
- verwijder rotatiegewricht in $O$.
- introduceer een schuifgewricht in $O$ in de $x$-richting, en een uitwendige kracht $F_{12x}$ in dezelfde richting.
- beschouw een virtuele verplaatsing $\delta x_{O}^*$ langs dat schuifgewricht.
- bereken de bijhorende $\delta r_{ix}^*, \delta r_{iy}^*$ en $\delta \phi_i^*$ via een kinematische snelheidsanalyse van het nieuwe mechanisme.

In dit specifieke geval vinden we deze virtuele verplaatsingen op het zicht:
$\delta r_{ix}^* = \delta x_{O}^*, \quad
\delta r_{iy}^* = 0, \quad
\delta \phi_i^* = 0.
$

Virtuele arbeid is uitgedrukt als:
$$
 F_{12x}  \delta x_{O}^* +
 \sum_{i=2}^4 \left[
 -m_i  \left( a_{ix}  \delta r_{ix} + a_{iy}  \delta r_{iy} \right)
 -I_{cgi}  \alpha_i \delta \phi_i^* \right] =0,
$$
of, door invullen van de virtuele verplaatsingen:
$$
 F_{12x}  \delta x_{O}^* - \sum_{i=2}^4 m_i  a_{ix}  \delta x_O^* =0
 \quad\Rightarrow\quad F_{12x} = \sum_{i=2}^4 m_i  a_{ix}.
$$

</div>


In [ ]:
F_x_check = np.zeros_like(t)
F_y_check = np.zeros_like(t)

for i in range(len(t)):
    F_x_check[i] = m2 * a2_x[i] + m3 * a3_x[i] + m6 * a6_x[i] + mDBC * aDBC_x[i] + mL * aL_x[i] + m7 * a7_x[i] + mEGF * aEGF_x[i]
    F_y_check[i] = m2 * a2_y[i] + m3 * a3_y[i] + m6 * a6_y[i] + mDBC * aDBC_y[i] + mL * aL_y[i] + m7 * a7_y[i] + mEGF * aEGF_y[i]

plt.figure()
plt.subplot(211)
plt.plot(t, F_x_check - (F_O_x + F_D_x))
plt.xlabel('t [s]')
plt.ylabel('Absolute error [W]')
plt.grid()
plt.title('check of total external force in x')

plt.subplot(212)
plt.plot(t, F_y_check - (F_O_y + F_D_y))
plt.xlabel('t [s]')
plt.ylabel('Absolute error [W]')
plt.grid()
plt.title('check of total external force in y')

plt.show()



Check van momentum via behoud van impuls:

\begin{aligned}
M_\text{shak}(t) &= -\sum_{i=2}^4 I_{cgi} \alpha_i + m_i (r_{ix} a_{iy} - r_{iy} a_{ix}).
\end{aligned}

In [ ]:
M_shake = np.zeros_like(t)
# Polaire traagheidsmomenten rond massacentrum voor alle staven en driehoeken
Icg2 = (1/12) * m2 * r_2**2 
IcgL = (1/12) * mL * L_1**2 
Icg3 = (1/12) * m3 * r_3**2
Icg6 = (1/12) * m6 * r_6**2
Icg7 = (1/12) * m7 * r_7**2
IcgDBC = moment_of_inertia_massacentrum(r_4, r_DC, theta_C, rho_triangle)
IcgEGF = moment_of_inertia_massacentrum(r_8, L_2, theta_I, rho_triangle)

# positievectoren van de zwaartepunten van alle staven en driehoeken
rcg2 = np.zeros((len(theta_2), 3))
rcg3 = np.zeros((len(theta_2), 3))
rcg6 = np.zeros((len(theta_2), 3))
rcg7 = np.zeros((len(theta_2), 3))
rcgDBC = np.zeros((len(theta_2), 3))
rcgL = np.zeros((len(theta_2), 3))
rcgEGF = np.zeros((len(theta_2), 3))

for i in range(len(t)):
    
    # Positie van het massacentrum van elke component tov punt O
    rcg2[i] = np.array([r_2/2 * np.cos(theta_2[i]), r_2/2 * np.sin(theta_2[i]), 0])
    rcg3[i] = A[i] + np.array([r_3/2 * np.cos(theta_3[i]), r_3/2 * np.sin(theta_3[i]), 0])
    rcg6[i] = A[i] + np.array([r_6/2 * np.cos(theta_6[i]), r_6/2 * np.sin(theta_6[i]), 0])
    rcg7[i] = C[i] + np.array([r_7/2 * np.cos(theta_7[i]), r_7/2 * np.sin(theta_7[i]), 0])
    rcgL[i] = D[i] + np.array([L_1/2 * np.cos(theta_f[i]), L_1/2 * np.sin(theta_f[i]), 0])
    # Voor de driehoeken moeten we de coördinaten van de hoekpunten gebruiken om het massacentrum te bepalen
    rcgDBC[i] = D[i] + centre_of_mass_triangle(D[i],B[i],C[i],D[i])
    rcgEGF[i] = E[i] + centre_of_mass_triangle(E[i],F[i],G[i],E[i])

    M_shake[i] = (Icg2 * ddtheta_2[i] + Icg3 * ddtheta_3[i] + Icg6 * ddtheta_6[i] + IcgDBC * ddtheta_4[i] + IcgL * ddtheta_f[i] + Icg7 * ddtheta_7[i] + IcgEGF * ddtheta_8[i] + 
    m2 * np.cross(rcg2[i], a2[i])[2] + m3 * np.cross(rcg3[i], a3[i])[2] + m6 * np.cross(rcg6[i], a6[i])[2] + mDBC * np.cross(rcgDBC[i], aDBC[i])[2] + mL * np.cross(rcgL[i], aL[i])[2] + m7 * np.cross(rcg7[i], a7[i])[2] + mEGF * np.cross(rcgEGF[i], aEGF[i])[2])
    
absolute_error_M = M_0 - D[:,1]*F_D_x + D[:,0]*F_D_y - M_shake # total external momentum

plt.plot(t, M_shake)
plt.xlabel('t [s]')
plt.ylabel('$absolute_error$ [Nm]')
plt.grid()
plt.title('shaking moment')
plt.show()


## Dynamica met ground reaction force

<div style="display: grid; place-items: center; place-content: center;">
 <div style="height: 30rem;">
 <img style="border: 0; display: inline-block;" height="100%"
      src="images\hyperstatisch VLD 1 been.png"
 >
</div>      

Bij het vrijmaken van de persoon merken we dat er in de werkelijkheid meerdere contactkrachten zijn tussen de persoon en het exoskelet. Omdat al deze krachten leiden tot een hyperstatisch systeem dat meer onbekende heeft dan vergelijkingen, wordt er voor de eenvoud enkel een contactkracht op de voeten beschouwd die een bepaalde fractie $\alpha$ van de contactkracht tussen de voet en het exoskelet is. Er geldt dus op het onderste punt van het exoskelet: 

$F_{G,res} = GRF - F_{voet} = \alpha * GRF$

Hieruit volgt dat:

$F_{voet} = (1 - \alpha)*GRF$

Uit deze vereenvoudiging kan $\alpha$ beschouwd worden als een ontwerpparamenter voor de fractie van de ground reaction force opgenomen door het exoskelet.  

Nu is het ook mogelijk om de resulterende wandelbeweging van de persoon te gaan berekenen obv de tweede wet van Newton om te zien wat de invloed is van de ondersteuningsparameter op de resulterende beweging van de persoon. Dit kan ook gelden als een validatie van het vereenvoudigde model.

In [ ]:
import os
os.path.exists("images/hyperstatisch VLD 1 been.png")


In [ ]:
import pandas as pd
import os
import numpy as np

def add_dataset_id(data):
    # for the GaitRec data we will define the dataset ID as 0
    data['DATASET_ID'] = 0
    return data

def make_unique_id(data1, data2):
    # SUBJECT_ID and SESSION_ID of data2 are changed to ensure their uniqueness (by adding the
    # maximum ID to the IDs of the GaitRec dataset)
    
    max_id = np.max(data1['SUBJECT_ID'].values)
    data2['SUBJECT_ID'] = data2['SUBJECT_ID']+max_id
    
    max_id = np.max(data1['SESSION_ID'].values)
    data2['SESSION_ID'] = data2['SESSION_ID']+max_id
    
    return data2 

def merge_data(data1, data2):
    # prior to merging we need to add a DATASET_ID for data2 and change the
    # SUBJECT_ID and SESSION_ID to ensure their uniqueness
    
    data2 = add_dataset_id(data2)
    data2 = make_unique_id(data1,data2)
    data = pd.concat([data1, data2], ignore_index=True, sort=False)
    return data


# Path to data
path = 'Walking Data/'

# Left lower extremity
GRF_F_V_PRO_left = pd.read_csv(os.path.join(path,'GRF_F_V_PRO_left.csv'))
# GRF_F_V_RAW_left = pd.read_csv(os.path.join(path,'GRF_F_V_RAW_left.csv'))

GRF_F_AP_PRO_left = pd.read_csv(os.path.join(path,'GRF_F_AP_PRO_left.csv'))
# GRF_F_AP_RAW_left = pd.read_csv(os.path.join(path,'GRF_F_AP_RAW_left.csv'))

row_y = GRF_F_V_PRO_left.iloc[3]

row_x = GRF_F_AP_PRO_left.iloc[0]








In [ ]:
# conditie nodig om te weten wanneer je in welk deel van de cyclus zit
# Dit moeten we dan linken aan theta_2
# conditie: horizontale snelheid van de enkel (punt G) is nul bij contact, de afgeleide is negatief (snelheid van positief naar negatief)


for i in range(len(t)):
    vG[i] = vE[i] + np.cross([0,0, dtheta_8[i]], G[i] - E[i]) # lineaire snelheid van punt G


t_contact = []
t_loss_contact = []
for i, t_i in enumerate(t):
    if i < len(t) - 1:
        if vG[i,0] > 0 and vG[i+1,0] < 0:
            print(f"Contact at time {t_i} seconds")
            t_contact.append(t_i)
        elif vG[i,0] < 0 and vG[i+1,0] > 0:
            print(f"Contact loss at time {t_i} seconds")
            t_loss_contact.append(t_i)

T_force = t_loss_contact[0] - t_contact[0]   # time vector, lengte is kleiner dan de data vector van de krachten, sampling time groter maken of krachten vector minder nauwkeurig maken

from scipy.interpolate import interp1d

force_y_data = row_y.filter(like="F_V_PRO").values
force_x_data = row_x.filter(like="F_AP_PRO").values

n_points = 101
t_force = np.linspace(t_contact[0], t_loss_contact[0], n_points)


t_force_active = np.arange(t_contact[0] - t_contact[0], t_loss_contact[0] - t_contact[0], Ts)  # time vector

force_y_func = interp1d(t_force, force_y_data, bounds_error=False, fill_value=0.0)
force_x_func = interp1d(t_force, force_x_data, bounds_error=False, fill_value=0.0)

force_y = force_y_func(t)
force_x = force_x_func(t)

T_cycle = (2 * np.pi) / np.abs(omega)
T_force = t_loss_contact[0] - t_contact[0]  # duration of contact phase

def force_x_func_cyclic(t_val):
    t_in_cycle = t_val % T_cycle  # where in the cycle are we? (0 to T_cycle)
    t_mapped = t_in_cycle  # shift to match interpolation domain
    if t_mapped < t_contact[0] or t_mapped > t_loss_contact[0]:
        return 0.0
    return float(force_x_func(t_mapped))

def force_y_func_cyclic(t_val):
    t_in_cycle = t_val % T_cycle  # where in the cycle are we? (0 to T_cycle)
    t_mapped = t_in_cycle  # shift to match interpolation domain
    if t_mapped < t_contact[0] or t_mapped > t_loss_contact[0]:
        return 0.0
    return float(force_y_func(t_mapped))


force_x_ext = np.vectorize(force_x_func_cyclic)(t)
force_y_ext = np.vectorize(force_y_func_cyclic)(t)

'''force_x_ext = np.zeros_like(t)
force_y_ext = np.zeros_like(t)

i_contact = np.argmin(np.abs(t - t_contact[0]))
i_loss_contact   = np.argmin(np.abs(t - t_loss_contact[0]))
i_end_cycle = int(np.round((np.pi*2/np.abs(omega))/Ts))

print(i_contact), print(i_loss_contact), print(i_end_cycle), print(t_force_active)

force_x_ext = np.tile(np.concatenate([np.zeros(i_contact), force_x_func(t_force_active), np.zeros(int(i_end_cycle - i_loss_contact))]), len(t_contact))
force_y_ext = np.tile(np.concatenate([np.zeros(i_contact), force_y_func(t_force_active), np.zeros(int(i_end_cycle - i_loss_contact))]), len(t_contact))'''








In [ ]:
g = 0 # momenteel nog geen zwaartekracht (als je wel toevoegd moet er een minteken voor)
heupmontage = 12 # gewicht van de heupmontage uit paper
m_persoon = 70 + heupmontage # gewicht van de persoon 
alpha = 0.5 # ondersteuningsparameter die bepaalt hoeveel procent van GRF op het exoskeleton komt. 

# Matrices voor krachten en momenten
F_O_x_wf = np.zeros_like(theta_2)
F_O_y_wf = np.zeros_like(theta_2)
F_A2_x_wf = np.zeros_like(theta_2)
F_A2_y_wf = np.zeros_like(theta_2)
F_A3_x_wf = np.zeros_like(theta_2)
F_A3_y_wf = np.zeros_like(theta_2)
F_A6_x_wf = np.zeros_like(theta_2)
F_A6_y_wf = np.zeros_like(theta_2)
F_B_x_wf = np.zeros_like(theta_2)
F_B_y_wf = np.zeros_like(theta_2)
F_E6_x_wf = np.zeros_like(theta_2)
F_E6_y_wf = np.zeros_like(theta_2)
F_D_x_wf = np.zeros_like(theta_2)
F_D_y_wf = np.zeros_like(theta_2)
F_DL_x_wf = np.zeros_like(theta_2)
F_DL_y_wf = np.zeros_like(theta_2)
F_EL_x_wf = np.zeros_like(theta_2)
F_EL_y_wf = np.zeros_like(theta_2)
F_C_x_wf = np.zeros_like(theta_2)
F_C_y_wf = np.zeros_like(theta_2)
F_F_x_wf = np.zeros_like(theta_2)
F_F_y_wf = np.zeros_like(theta_2)
F_DT_x_wf = np.zeros_like(theta_2)
F_DT_y_wf = np.zeros_like(theta_2)
F_ET_x_wf = np.zeros_like(theta_2)
F_ET_y_wf = np.zeros_like(theta_2)
M_0_wf = np.zeros_like(theta_2)

force_x_verdeeld = alpha*force_x_ext*m_persoon*10
force_y_verdeeld = alpha*force_y_ext*m_persoon*10

# Oplossen van kracht en moment evenwichten voor elke tijdstap
for i in range(len(t)):
        E_x, E_y, F_x, F_y, G_x, G_y = coordinaten_tov_massacentrum_driehoek(L_2, r_8, theta_I, theta_8[i])
#FO_x, FO_y, FA2_x, FA2_y, FA3_x, FA3_y, FA6_x, FA6_y, FB_x, FB_y, FE6_x, FE6_y, FD_x, FD_y, FDL_x, FDL_y, FEL_x, FEL_y, FC_x, FC_y, FF_x, FF_y, FDT_x, FDT_y, FET_x, FET_y, M_0 = np.linalg.solve(A, b)
        AF = np.array([[1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],     #staaf R2
                      [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                      [0, 0, -r_2 * np.sin(theta_2[i]), r_2 * np.cos(theta_2[i]), 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
                      
                      [0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],    #staaf R3
                      [0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                      [0, 0, 0, 0, 0.5 * r_3 * np.sin(theta_3[i]), -0.5 * r_3 * np.cos(theta_3[i]), 0, 0, -0.5 * r_3 * np.sin(theta_3[i]), 0.5 * r_3 * np.cos(theta_3[i]), 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                      
                      [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],    #staaf R6
                      [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                      [0, 0, 0, 0, 0, 0, 0.5 * r_6 * np.sin(theta_6[i]), -0.5 * r_6 * np.cos(theta_6[i]), 0, 0, -0.5 * r_6 * np.sin(theta_6[i]), 0.5 * r_6 * np.cos(theta_6[i]), 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],

                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],    #staaf L1
                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -L_1 * np.sin(theta_f[i]), L_1 * np.cos(theta_f[i]), 0, 0, 0, 0, 0, 0, 0, 0, 0],

                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0],    #staaf R7
                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0],
                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.5 * r_7 * np.sin(theta_7[i]), -0.5 * r_7 * np.cos(theta_7[i]), -0.5 * r_7 * np.sin(theta_7[i]), 0.5 * r_7 * np.cos(theta_7[i]), 0, 0, 0, 0, 0],
                      
                      [0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 1, 0, 0, 0, 0],    #driehoek DBC
                      [0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 1, 0, 0, 0],
                      [0, 0, 0, 0, 0, 0, 0, 0, r_4 * np.sin(theta_4[i]), -r_4 * np.cos(theta_4[i]), 0, 0, 0, 0, 0, 0, 0, 0, r_DC * np.sin(theta_4[i]+theta_C), -r_DC * np.cos(theta_4[i]+theta_C), 0, 0, 0, 0, 0, 0, 0],

                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 1, 0, 0],    #driehoek EGF
                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 1, 0],
                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, F_y, -F_x, 0, 0, -E_y, E_x, 0],
                
                      [0, 0, -1, 0, -1, 0, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], #knoop A
                      [0, 0, 0, -1, 0, -1, 0, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],      

                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, -1, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 0], #knoop D
                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, -1, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0],

                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 0, 0, -1, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0], #knoop E
                      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 0, 0, -1, 0, 0, 0, 0, 0, 0, 0, -1, 0]             
                      ])

        BF = np.array([m2 * a2_x[i],
                      m2 * (a2_y[i]+g),
                      J2 * ddtheta_2[i],

                      m3 * a3_x[i],
                      m3 * (a3_y[i]+g),
                      J3 * ddtheta_3[i],

                      m6 * a6_x[i],
                      m6 * (a6_y[i]+g),
                      J6 * ddtheta_6[i],

                      mL * aL_x[i],
                      mL * (aL_y[i]+g),
                      J_L * ddtheta_f[i],

                      m7 * a7_x[i],
                      m7 * (a7_y[i]+g),
                      J7 * ddtheta_7[i],

                      mDBC * aDBC_x[i],
                      mDBC * (aDBC_y[i]+g),
                      J_DBC * ddtheta_4[i],

                      mEGF * aEGF_x[i] - force_x_verdeeld[i],
                      mEGF * (aEGF_y[i]+g) - force_y_verdeeld[i],
                      J_EGF * ddtheta_8[i] - ((G[i,0] - rcgEGF[i,0])*force_y_verdeeld[i] - (G[i,1] - rcgEGF[i,1])*force_x_verdeeld[i]), 
                                    
                      0,
                      0,
                      
                      0,
                      0,
                      
                      0,
                      0])

        x_wf = np.linalg.lstsq(AF, BF, rcond=None)[0]

        # Save results
        F_O_x_wf[i] = x_wf[0]
        F_O_y_wf[i] =  x_wf[1]
        F_A2_x_wf[i] = x_wf[2]
        F_A2_y_wf[i] = x_wf[3]
        F_A3_x_wf[i] = x_wf[4]
        F_A3_y_wf[i] = x_wf[5]
        F_A6_x_wf[i] = x_wf[6]
        F_A6_y_wf[i] = x_wf[7]
        F_B_x_wf[i] = x_wf[8]
        F_B_y_wf[i] = x_wf[9]
        F_E6_x_wf[i] = x_wf[10]
        F_E6_y_wf[i] = x_wf[11]
        F_D_x_wf[i] = x_wf[12]
        F_D_y_wf[i] = x_wf[13]
        F_DL_x_wf[i] = x_wf[14]
        F_DL_y_wf[i] = x_wf[15]
        F_EL_x_wf[i] = x_wf[16]
        F_EL_y_wf[i] = x_wf[17]
        F_C_x_wf[i] = x_wf[18]
        F_C_y_wf[i] = x_wf[19]
        F_F_x_wf[i] = x_wf[20]
        F_F_y_wf[i] = x_wf[21]
        F_DT_x_wf[i] = x_wf[22]
        F_DT_y_wf[i] = x_wf[23]
        F_ET_x_wf[i] = x_wf[24]
        F_ET_y_wf[i] = x_wf[25]
        M_0_wf[i] = x_wf[26]


In [ ]:
plt.figure()
plt.subplot(221)
plt.plot(t, F_O_x_wf)
plt.plot(t, F_O_y_wf)
plt.xlabel('t [s]')
plt.ylabel(r'$F_O$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.subplot(222)
plt.plot(t, F_A2_x_wf)
plt.plot(t, F_A2_y_wf)
plt.xlabel('t [s]')
plt.ylabel(r'$F_{A2}$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.subplot(223)
plt.plot(t, F_A3_x_wf)
plt.plot(t, F_A3_y_wf)
plt.xlabel('t [s]')
plt.ylabel(r'$F_{A3}$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.subplot(224)
plt.plot(t, F_A6_x_wf)
plt.plot(t, F_A6_y_wf)
plt.xlabel('t [s]')
plt.ylabel(r'$F_{A6}$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.tight_layout()
plt.show()


plt.subplot(221)
plt.plot(t, F_B_x_wf)
plt.plot(t, F_B_y_wf)
plt.xlabel('t [s]')
plt.ylabel(r'$F_B$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.subplot(222)
plt.plot(t, F_E6_x_wf)
plt.plot(t, F_E6_y_wf)
plt.xlabel('t [s]')
plt.ylabel(r'$F_{E6}$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.subplot(223)
plt.plot(t, F_D_x_wf)
plt.plot(t, F_D_y_wf)
plt.xlabel('t [s]')
plt.ylabel(r'$F_{D}$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.subplot(224)
plt.plot(t, F_DL_x_wf)
plt.plot(t, F_DL_y_wf)
plt.xlabel('t [s]')
plt.ylabel(r'$F_{DL}$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.tight_layout()
plt.show()

plt.subplot(221)
plt.plot(t, F_EL_x_wf)
plt.plot(t, F_EL_y_wf)
plt.xlabel('t [s]')
plt.ylabel(r'$F_{EL}$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.subplot(222)
plt.plot(t, F_C_x_wf)
plt.plot(t, F_C_y_wf)
plt.xlabel('t [s]')
plt.ylabel(r'$F_{C}$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.subplot(223)
plt.plot(t, F_F_x_wf)
plt.plot(t, F_F_y_wf)
plt.xlabel('t [s]')
plt.ylabel(r'$F_{F}$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.subplot(224)
plt.plot(t, F_DT_x_wf)
plt.plot(t, F_DT_y_wf)
plt.xlabel('t [s]')
plt.ylabel(r'$F_{DT}$ [N]')
plt.legend(["x", "y"])
plt.grid()

plt.tight_layout()
plt.show()


plt.subplot(221)
plt.plot(t, F_ET_x_wf)
plt.plot(t, F_ET_y_wf)
plt.xlabel('t [s]')
plt.ylabel(r'$F_{ET}$ [N]')
plt.legend(["x", "y"])
plt.grid()


plt.tight_layout()
plt.show()



In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(t, M_0_wf)
plt.title("Aandrijfmoment M_0")
plt.xlabel("t")
plt.ylabel("M_0")
plt.grid()
plt.show()

In [ ]:
plt.figure()
plt.plot(t, F_O_x_wf, label="F_O_x")
plt.plot(t, F_O_y_wf, label="F_O_y")
plt.legend()
plt.title("Reactiekracht in O")
plt.grid()
plt.show()

In [ ]:
F_O_wf = np.sqrt(F_O_x_wf**2 + F_O_y_wf**2) # onbalanskrachten aan de motor

plt.figure()
plt.plot(t, F_O_wf)
plt.title("Resultante kracht in O")
plt.grid()
plt.show()

Variatie van kinetische energie om het vermogen te bepalen

In [ ]:
M_ext_with_force = np.zeros_like(t)

for i in range(len(t)):
    M_ext_with_force[i] = (m2*np.dot(v1[i], a2[i]) + m3*np.dot(v2[i], a3[i]) + m6*np.dot(v3[i], a6[i]) + mDBC*np.dot(v4[i], aDBC[i]) + mL*np.dot(v5[i], aL[i]) + m7*np.dot(v6[i], a7[i]) + mEGF*np.dot(v7[i], aEGF[i]) +
                (1/12) * m2 *r_2 **2 *dtheta_2[i]*ddtheta_2[i] + J3*dtheta_3[i]*ddtheta_3[i] + J6*dtheta_6[i]*ddtheta_6[i] + moment_of_inertia_massacentrum(r_4, r_DC, theta_C, rho_triangle)*dtheta_4[i]*ddtheta_4[i] +(1/12) * mL * L_1**2 * dtheta_f[i]*ddtheta_f[i] + J7*dtheta_7[i]*ddtheta_7[i] + J_EGF*dtheta_8[i]*ddtheta_8[i]
                - np.dot([force_x_verdeeld[i], force_y_verdeeld[i], 0], vG[i])) / dtheta_2[i] 
    

plt.plot(t, M_ext_with_force)
plt.xlabel('t [s]')
plt.ylabel('$M_{ext}$ [Nm]')
plt.grid()
plt.title('Variation of kinetic energy')
plt.show()

plt.plot(t, M_ext_with_force - M_0_wf)
plt.xlabel('t [s]')
plt.ylabel('Absolute error [W]')
plt.grid()
plt.title('check of external moment')
plt.show()

plt.plot(t, -M_ext_with_force*dtheta_2)
plt.xlabel('t [s]')
plt.ylabel('P [W]')
plt.grid()
plt.title('power of the driver')
plt.show()

Shaking forces aan de heup met externe krachten. Er wordt een onderscheid gemaakt tussen de shaking forces owv de inertie van het mechanisme en de totale externe krachten waneer er een kracht wordt aangelegd. Als er een externe kracht is zal dit de krachten op het frame en de heup van de persoon beinvloeden. 

In [ ]:
F_ext_x_with_GRF = F_D_x_wf + F_O_x_wf + force_x_verdeeld
F_ext_y_with_GRF = F_D_y_wf + F_O_y_wf + force_y_verdeeld
M_ext_with_GRF = M_0_wf - (-(-D[:,1])*F_O_wf + (-D[:,0])*F_O_y_wf) + (- (G[:,1] - D[:,1])*force_x_verdeeld + (G[:,0] - D[:,0])*force_y_verdeeld)


F_shake_x = -F_ext_x_with_GRF + force_x_verdeeld
F_shake_y = -F_ext_y_with_GRF + force_y_verdeeld
M_shake = -M_ext_with_GRF + ( - (G[:,1] - D[:,1])*force_x_verdeeld + (G[:,0] - D[:,0])*force_y_verdeeld)


F_ext_x = F_D_x + F_O_x + force_x_verdeeld
F_ext_y = F_D_y + F_O_y + force_y_verdeeld
M_ext_D = M_0 + (-(-D[:,1])*F_O_x + (-D[:,0])*F_O_y) - (G[:,1] - D[:,1])*force_x_verdeeld + (G[:,0] - D[:,0])*force_y_verdeeld

F_shake_x_inertial = -F_ext_x + force_x_verdeeld
F_shake_y_inertial = -F_ext_y + force_y_verdeeld
M_shake_inertial = -M_ext_D + ( - (G[:,1] - D[:,1])*force_x_verdeeld + (G[:,0] - D[:,0])*force_y_verdeeld)


# Plot 1: shaking force in X-Direction
plt.figure(figsize=(8, 4))
plt.plot(t, F_shake_x, label = 'Shaking force with external force')
plt.plot(t, F_shake_x_inertial, label = 'Shaking force due to inertia')
plt.xlabel('t [s]')
plt.ylabel('Force [N]')
plt.title('Shaking force in X-Direction')
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

# Plot 2: shaking force in Y-Direction
plt.figure(figsize=(8, 4))
plt.plot(t, F_shake_y, label = 'Shaking force with external force')
plt.plot(t, F_shake_y_inertial, label = 'Shaking force due to inertia')
plt.xlabel('t [s]')
plt.ylabel('Force [N]')
plt.title('Shaking force in Y-Direction')
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

# Plot 3: shaking torque around joint D
plt.figure(figsize=(8, 4))
plt.plot(t, M_shake, label = 'Shaking moment with external force')
plt.plot(t, M_shake_inertial, label = 'Shaking moment due to inertia')
plt.xlabel('t [s]')
plt.ylabel('Moment [Nm]')
plt.title('Shaking torque around joint D')
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

Onbalans krachten owv het mechanisme zelf
- De kracht is kwadratisch afhankelijk van de wandelsnelheid F = m*w^2*r (versimpeld). Hieruit volgt dat de pure onbalans krachten sterk afhankelijk zijn van de snelheid waarmee gewandeld wordt. 

In [ ]:
F_ext_x = F_D_x + F_O_x + force_x_verdeeld
F_ext_y = F_D_y + F_O_y + force_y_verdeeld

# moment dat op de heup van de persoon terechtkomt.
M_ext_D = M_0 + (-(-D[:,1])*F_O_x + (-D[:,0])*F_O_y) - (G[:,1] - D[:,1])*force_x_verdeeld + (G[:,0] - D[:,0])*force_y_verdeeld

F_shake_x = -F_ext_x + force_x_verdeeld
F_shake_y = -F_ext_y + force_y_verdeeld
M_shake = -M_ext_D + ( - (G[:,1] - D[:,1])*force_x_verdeeld + (G[:,0] - D[:,0])*force_y_verdeeld)


# Plot 1: shaking force in X-Direction
plt.figure(figsize=(8, 4))
plt.plot(t, F_shake_x)
plt.xlabel('t [s]')
plt.ylabel('Force [N]')
plt.title('Shaking force in X-Direction')
plt.grid()
plt.tight_layout()
plt.show()

# Plot 2: shaking force in Y-Direction
plt.figure(figsize=(8, 4))
plt.plot(t, F_shake_y)
plt.xlabel('t [s]')
plt.ylabel('Force [N]')
plt.title('Shaking force in Y-Direction')
plt.grid()
plt.tight_layout()
plt.show()

# Plot 3: shaking torque around joint D
plt.figure(figsize=(8, 4))
plt.plot(t, M_shake)
plt.xlabel('t [s]')
plt.ylabel('Moment [Nm]')
plt.title('Shaking torque around joint D')
plt.grid()
plt.tight_layout()
plt.show()

## Design of the flywheel

In [ ]:
M_to_balance = M_0_wf
M_avg = np.mean(M_to_balance)
omega = 2.1
i_end_cycle = int(np.round((np.pi*2/np.abs(omega))/Ts))
theta = -theta_2[:i_end_cycle]

# Torque plot
plt.figure()
plt.plot(theta, M_to_balance[:i_end_cycle], label='Torque T(s)')
plt.axhline(M_avg, color='red', linestyle='--', label='Average Torque')
plt.xlabel('theta [rad]')
plt.ylabel('M [Nm]')
plt.grid()
plt.title('Torque')
plt.legend()
plt.tight_layout()
plt.show()

theta = -theta_2[:i_end_cycle]
dtheta = np.gradient(theta)
print(dtheta)
A_theta = np.cumsum((M_to_balance[:i_end_cycle] - M_avg)*dtheta)

# Energy deviation plot
plt.figure()
plt.plot(theta, A_theta, label='A(θ): Energy deviation', color='purple')
plt.axhline(0, color='black', linestyle='--')
plt.xlabel('t [s]')
plt.ylabel('A(s) [J]')
plt.title('Cumulative Energy Deviation A(theta)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

A_max = np.max(A_theta) - np.min(A_theta)


# Flywheel inertia
omega_max = omega * 1.1
omega_min = omega * 0.9
K = (omega_max - omega_min) / omega
inertia_tot = A_max / (K * omega**2)
print(f"I = {inertia_tot:.4f} kgm²")

# Inertia vs K plot
K_values = np.linspace(0.01, 0.5, 500)
inertia_tot_list = A_max / (K_values * omega**2)

plt.figure(figsize=(10, 6))
plt.plot(K_values, inertia_tot_list, label='Total needed inertia')
K_line = 0.2
I_line = A_max / (K_line * omega**2)
plt.axvline(x=K_line, color='red', linestyle='--', label=f'K = {K_line:.2f}')
plt.scatter([K_line], [I_line], color='red')
plt.text(K_line + 0.01, I_line, f'I = {I_line:.2f}', color='red')
plt.title('Needed Inertia vs Fluctuation Coefficient K')
plt.xlabel('Fluctuation Coefficient K')
plt.ylabel('Total Inertia (kgm²)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

inertia_flywheel = inertia_tot

# Inertia vs omega plot
omega_values = np.linspace(0.1, 4, 500)
inertia_vs_omega = np.zeros_like(omega_values)
for i, w in enumerate(omega_values):
    theta_i = w*t[:i_end_cycle]
    dtheta_i = np.gradient(theta_i)
    A_theta_i = np.cumsum((M_to_balance[:i_end_cycle] - M_avg)*dtheta_i)
    A_max_i = np.max(A_theta) - np.min(A_theta_i)
    K = 0.2
    inertia_vs_omega[i] = A_max_i / (K*w**2)

plt.figure()
plt.plot(omega_values, inertia_vs_omega)
plt.title('Inertia for specific pulsation with K = 0.2')
plt.xlabel('omega')
plt.ylabel('Inertia')
plt.show()



Plotten van omega in functie van het moment

In [ ]:
omega_O = np.sqrt(2*np.abs(A_theta))
print(omega_O)
plt.plot(t[:i_end_cycle], omega_O)
plt.title('Pulsation of driving axis for inertia equal to one')
plt.xlabel('time')
plt.ylabel('omega')
plt.show()

---
# Motor dimensionering

We dimensioneren de motor op basis van het koppelprofiel uit de inverse dynamica. 
Het mechanisme heeft 1 DOF, dus alle beweging wordt opgelegd via de inputhoek θ₂. 
Daarom is het aandrijfkoppel M₀ rechtstreeks het koppel dat op de inputas geleverd moet worden.

We gebruiken:
- $M_O$ voor het inertiële koppel
- $M_{0,wf}$ voor het koppel met externe kracht op de voet
- P = $M_0$ · $\omega_2$ voor het mechanisch vermogen
- het piekkoppel voor motorselectie
- het gemiddeld/positief vermogen voor energieverbruik

In [ ]:
# ============================================================
# Motor dimensionering op basis van M_0_wf
# ============================================================

"""
We gebruiken het eerder berekende aandrijfmoment M_0_wf 
(= moment aan de inputlink wanneer de grondreactiekracht wordt meegenomen)
"""

# Een volledige cyclus selecteren
omega_driver = np.mean(np.abs(dtheta_2))  # [rad/s]
i_end_cycle = int(np.round((2*np.pi / omega_driver) / Ts))

t_motor = t[:i_end_cycle] - t[0]
theta_motor = -theta_2[:i_end_cycle]

M_without_GRF = M_0[:i_end_cycle]
M_with_GRF = M_0_wf[:i_end_cycle]
omega_motor = dtheta_2[:i_end_cycle]

# Zelfde tekenconventie als in de eerdere vermogensplot:
# in jullie notebook werd P = -M*dtheta_2 gebruikt
P_mech = -M_with_GRF * omega_motor

# Maximaal koppel
M_peak_abs = np.max(np.abs(M_with_GRF))
i_peak = np.argmax(np.abs(M_with_GRF))

# RMS-koppel
M_rms = np.sqrt(np.mean(M_with_GRF**2))

# Vermogen
P_peak_abs = np.max(np.abs(P_mech))
P_mean_abs = np.mean(np.abs(P_mech))
P_mean_pos = np.mean(np.maximum(P_mech, 0))

# Energie per cyclus
E_net_cycle = np.trapezoid(P_mech, t_motor)
E_abs_cycle = np.trapezoid(np.abs(P_mech), t_motor)
E_pos_cycle = np.trapezoid(np.maximum(P_mech, 0), t_motor)

print("Motorvereisten op basis van M_0_wf")
print("----------------------------------")
print(f"Maximaal absoluut koppel: {M_peak_abs:.3f} Nm")
print(f"Tijdstip max koppel: {t_motor[i_peak]:.3f} s")
print(f"Inputhoek max koppel: {theta_motor[i_peak]:.3f} rad")
print(f"RMS-koppel: {M_rms:.3f} Nm")
print()
print(f"Maximaal absoluut vermogen: {P_peak_abs:.3f} W")
print(f"Gemiddeld absoluut vermogen: {P_mean_abs:.3f} W")
print(f"Gemiddeld positief vermogen: {P_mean_pos:.3f} W")
print()
print(f"Netto energie per cyclus: {E_net_cycle:.3f} J")
print(f"Absolute energie per cyclus: {E_abs_cycle:.3f} J")
print(f"Positieve energie per cyclus: {E_pos_cycle:.3f} J")

In [ ]:
# ============================================================
# Plot aandrijfkoppel met en zonder grondreactiekracht
# ============================================================

plt.figure(figsize=(8, 4))
plt.plot(theta_motor, M_without_GRF, label=r"$M_0$ zonder grondreactie")
plt.plot(theta_motor, M_with_GRF, label=r"$M_{0,wf}$ met grondreactie")

plt.axvline(theta_motor[i_peak], linestyle="--", label="positie max |koppel|")
plt.scatter(theta_motor[i_peak], M_with_GRF[i_peak])

plt.xlabel(r"inputhoek $\theta_2$ [rad]")
plt.ylabel("aandrijfmoment [Nm]")
plt.title("Aandrijfmoment met en zonder grondreactiekracht")
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

print(f"Maximaal absoluut koppel met grondreactie: {M_peak_abs:.3f} Nm")

Negatief vermogen wordt hier niet als geregenereerde energie beschouwd. Voor energieverbruik gebruiken we daarom het absoluut mechanisch vermogen.

## Gewenst snelheidsbereik

Voor de koppeling tussen wandelsnelheid en rotatiesnelheid gebruiken we de referentiewaarde uit de paper. Daarin wordt een wandelsnelheid van 1,5 km/h bereikt met een rotatiesnelheid van 20 rpm aan de inputlink. We nemen daarom aan dat de wandelsnelheid lineair schaalt met de input-rpm:

$v = 0{,}075 \cdot n$

met $v$ in km/h en $n$ in rpm. De hoeksnelheid volgt uit:

$\omega = n \cdot \frac{2\pi}{60}$

Het gewenste snelheidsbereik van 1 tot 3 km/h komt dus overeen met ongeveer 13,3 tot 40 rpm aan de inputlink.

In [ ]:

# Referentie uit de paper:
# 1.5 km/h komt overeen met 20 rpm aan de inputlink
v_ref = 1.5       # [km/h]
rpm_ref = 20.0    # [rpm]

def kmh_to_rpm(v_kmh):
    return rpm_ref * v_kmh / v_ref

def rpm_to_kmh(rpm):
    return v_ref * rpm / rpm_ref

def rpm_to_omega(rpm):
    return rpm * 2*np.pi / 60

speed_table = pd.DataFrame([
    {
        "wandelsnelheid [km/h]": 1.0,
        "input-rpm [rpm]": kmh_to_rpm(1.0),
        "hoeksnelheid [rad/s]": rpm_to_omega(kmh_to_rpm(1.0)),
        "interpretatie": "traagste gewenste werking"
    },
    {
        "wandelsnelheid [km/h]": 1.5,
        "input-rpm [rpm]": kmh_to_rpm(1.5),
        "hoeksnelheid [rad/s]": rpm_to_omega(kmh_to_rpm(1.5)),
        "interpretatie": "referentie uit paper"
    },
    {
        "wandelsnelheid [km/h]": 3.0,
        "input-rpm [rpm]": kmh_to_rpm(3.0),
        "hoeksnelheid [rad/s]": rpm_to_omega(kmh_to_rpm(3.0)),
        "interpretatie": "snelste gewenste werking"
    },
])

display(speed_table.round(3))

rpm_min_required = kmh_to_rpm(1.0)
rpm_nominal_required = kmh_to_rpm(1.5)
rpm_max_required = kmh_to_rpm(3.0)

print(f"Gewenst outputbereik motor: {rpm_min_required:.3f} rpm tot {rpm_max_required:.3f} rpm")

## Vergelijking van kandidaatmotoren

De motor wordt geselecteerd op basis van het eerder berekende koppelprofiel $M_{0,wf}$, dus met grondreactiekracht. We vergelijken de motoren met:

- het maximale absolute koppel, inclusief veiligheidsfactor;
- het RMS-koppel, als indicatie voor de thermische belasting;
- het gewenste snelheidsbereik van 1 tot 3 km/h;
- het gemiddeld positief mechanisch vermogen, aangezien we geen regeneratie aannemen;
- massa, kostprijs en praktische integratie.

De drie gekozen opties zijn bewust verschillend:
1. een DC-servo met planetaire overbrenging: licht en geschikt voor batterijvoeding;
2. een AC-servo met planetaire overbrenging: nauwkeurig en sterk regelbaar, maar minder geschikt voor draagbare voeding;
3. een industriële wormwielmotor: robuust en eenvoudig, maar zwaar.

In [ ]:
# ============================================================
# Vergelijking kandidaatmotoren
# ============================================================

# Waarden uit eigen analyse
# Deze variabelen komen uit de vorige motorcel:
# M_peak_abs, M_rms, P_mean_pos, E_pos_cycle

safety_factor = 1.5
electricity_price = 0.107      # [EUR/kWh], Belgische energieprijs volgens opgave
usd_to_eur = 1 / 1.1738        # ECB referentie: 1 EUR = 1.1738 USD

M_peak_required = safety_factor * M_peak_abs

# Efficiënties:
# Voor de planetaire gearbox gebruiken we 94% uit de datasheet.
# Voor motor + controller nemen we een ruwe schatting, omdat dit werkpunt sterk afhangt van regeling en belasting.
eta_motor_controller_servo = 0.80
eta_planetary = 0.94

# Voor de wormwielmotor nemen we een conservatieve totaalschatting.
# Deze waarde is bewust lager door hogere verliezen in wormoverbrengingen.
eta_total_worm = 0.75

motor_candidates = pd.DataFrame([
    {
        "optie": "A",
        "motor": "DC-servo + planetaire gearbox",
        "voorbeeld": "StepperOnline DSY-C200L2A2-M17S + HTG60-G50",
        "voeding": "24-70 VDC",
        "type": "servo, batterijvriendelijk",
        "motorvermogen [W]": 200,
        "motorkoppel nominaal [Nm]": 0.637,
        "motorkoppel piek [Nm]": 1.91,
        "motor rpm nominaal [rpm]": 3000,
        "motor rpm piek [rpm]": 4000,
        "reductie [-]": 50,
        "gearbox rendement [-]": 0.94,
        "gearbox max continu [Nm]": 45,
        "gearbox max momentaan [Nm]": 90,
        "geschat totaal rendement [-]": eta_motor_controller_servo * eta_planetary,
        "gewicht [kg]": 2.00 + 1.81,
        "kost motor [USD]": 117.68,
        "kost gearbox [USD]": 100.63,
        "voordeel": "licht, laagspanning, goed regelbaar",
        "nadeel": "CANopen/servo-aansturing complexer"
    },
    {
        "optie": "B",
        "motor": "AC-servo + planetaire gearbox",
        "voorbeeld": "StepperOnline A6-EC400H2A1-M17 + HTG60-G50",
        "voeding": "220 VAC",
        "type": "servo, hoge precisie",
        "motorvermogen [W]": 400,
        "motorkoppel nominaal [Nm]": 1.27,
        "motorkoppel piek [Nm]": 4.45,
        "motor rpm nominaal [rpm]": 3000,
        "motor rpm piek [rpm]": 6000,
        "reductie [-]": 50,
        "gearbox rendement [-]": 0.94,
        "gearbox max continu [Nm]": 45,
        "gearbox max momentaan [Nm]": 90,
        "geschat totaal rendement [-]": eta_motor_controller_servo * eta_planetary,
        "gewicht [kg]": 3.17 + 1.81,
        "kost motor [USD]": 112.05,
        "kost gearbox [USD]": 100.63,
        "voordeel": "beste regelbaarheid en precisie",
        "nadeel": "220 VAC, minder geschikt voor draagbaar systeem"
    },
])

# Uitgaand continu koppel
motor_candidates["continu koppel output [Nm]"] = np.minimum(
    motor_candidates["motorkoppel nominaal [Nm]"] 
    * motor_candidates["reductie [-]"] 
    * motor_candidates["gearbox rendement [-]"],
    motor_candidates["gearbox max continu [Nm]"]
)

# Uitgaand piek/startkoppel
motor_candidates["start-/piekkoppel output [Nm]"] = np.minimum(
    motor_candidates["motorkoppel piek [Nm]"] 
    * motor_candidates["reductie [-]"] 
    * motor_candidates["gearbox rendement [-]"],
    motor_candidates["gearbox max momentaan [Nm]"]
)

# Uitgaande rpm
motor_candidates["snelste rpm continu [rpm]"] = (
    motor_candidates["motor rpm nominaal [rpm]"] / motor_candidates["reductie [-]"]
)

motor_candidates["snelste snelheid continu [km/h]"] = motor_candidates[
    "snelste rpm continu [rpm]"
].apply(rpm_to_kmh)

motor_candidates["snelste rpm piek [rpm]"] = (
    motor_candidates["motor rpm piek [rpm]"] / motor_candidates["reductie [-]"]
)

motor_candidates["snelste snelheid piek [km/h]"] = motor_candidates[
    "snelste rpm piek [rpm]"
].apply(rpm_to_kmh)

# Traagste werkpunt binnen gewenst bereik
motor_candidates["traagste rpm gewenst [rpm]"] = rpm_min_required
motor_candidates["traagste snelheid gewenst [km/h]"] = rpm_to_kmh(rpm_min_required)

# Checks
motor_candidates["piekmarge [-]"] = (
    motor_candidates["start-/piekkoppel output [Nm]"] / M_peak_required
)

motor_candidates["RMS-marge [-]"] = (
    motor_candidates["continu koppel output [Nm]"] / M_rms
)

motor_candidates["voldoet piekkoppel?"] = (
    motor_candidates["start-/piekkoppel output [Nm]"] >= M_peak_required
)

motor_candidates["voldoet RMS-koppel?"] = (
    motor_candidates["continu koppel output [Nm]"] >= M_rms
)

motor_candidates["voldoet snelheid?"] = (
    motor_candidates["snelste rpm continu [rpm]"] >= rpm_max_required
)

# Energieverbruik
motor_candidates["geschat elektrisch vermogen [W]"] = (
    P_mean_pos / motor_candidates["geschat totaal rendement [-]"]
)

motor_candidates["elektrische energie per cyclus [J]"] = (
    E_pos_cycle / motor_candidates["geschat totaal rendement [-]"]
)

motor_candidates["energie per uur [kWh]"] = (
    motor_candidates["geschat elektrisch vermogen [W]"] / 1000
)

motor_candidates["energiekost per uur [EUR/h]"] = (
    motor_candidates["energie per uur [kWh]"] * electricity_price
)

motor_candidates["energiekost per 100 uur [EUR]"] = (
    100 * motor_candidates["energiekost per uur [EUR/h]"]
)

# Kost
motor_candidates["kost totaal [USD]"] = (
    motor_candidates["kost motor [USD]"] + motor_candidates["kost gearbox [USD]"]
)

motor_candidates["kost totaal [EUR]"] = (
    motor_candidates["kost totaal [USD]"] * usd_to_eur
)

motor_candidates["globaal bruikbaar?"] = (
    motor_candidates["voldoet piekkoppel?"]
    & motor_candidates["voldoet RMS-koppel?"]
    & motor_candidates["voldoet snelheid?"]
)

requirements_table = pd.DataFrame([{
    "max |M_0,wf| [Nm]": M_peak_abs,
    "safety factor [-]": safety_factor,
    "vereist piekkoppel incl. SF [Nm]": M_peak_required,
    "RMS-koppel [Nm]": M_rms,
    "gemiddeld positief mechanisch vermogen [W]": P_mean_pos,
    "positieve energie per cyclus [J]": E_pos_cycle,
    "gewenst snelheidsbereik [rpm]": f"{rpm_min_required:.3f} - {rpm_max_required:.3f}",
    "gewenst snelheidsbereik [km/h]": "1.000 - 3.000",
    "elektriciteitsprijs [EUR/kWh]": electricity_price,
}])

print("Eisen uit eigen analyse:")
display(requirements_table)

columns_to_show = [
    "optie",
    "motor",
    "voeding",
    "continu koppel output [Nm]",
    "start-/piekkoppel output [Nm]",
    "piekmarge [-]",
    "RMS-marge [-]",
    "snelste rpm continu [rpm]",
    "snelste snelheid continu [km/h]",
    "geschat elektrisch vermogen [W]",
    "elektrische energie per cyclus [J]",
    "energie per uur [kWh]",
    "energiekost per 100 uur [EUR]",
    "gewicht [kg]",
    "kost totaal [EUR]",
    "voordeel",
    "nadeel",
]

display(motor_candidates[columns_to_show].round(3))

Omdat beide opties ruim voldoen aan het vereiste piekkoppel met veiligheidsfactor, wordt de finale keuze vooral bepaald door voeding, gewicht, regelbaarheid en kostprijs. Voor ons ontwerp kiezen we de DC-servo als voorkeursoplossing.

## Validatie Motorselectie
### Gevoeligheid aan wandelsnelheid

Na de motorselectie controleren we hoe de vereiste koppels, vermogens en energieverbruik veranderen bij verschillende wandelsnelheden. We bekijken 1 km/h, 2 km/h en 3 km/h.

Volgens de referentiepaper komt 1,5 km/h overeen met 20 rpm aan de inputlink. We gebruiken deze verhouding om wandelsnelheid naar toerental om te rekenen.

Voor de schaling van het koppel maken we de volgende vereenvoudiging:

$M_{0,wf} = M_\text{zonder grondreactie} + M_\text{grondreactie}$

waarbij we aannemen dat:
- de inertiële component ongeveer schaalt met $k^2$, omdat versnellingen kwadratisch toenemen met snelheid;
- de grondreactiecomponent ongeveer gelijk blijft voor hetzelfde bewegingspatroon;
- de hoeksnelheid schaalt met $k$.

Hierbij is $k$ de snelheidsfactor ten opzichte van de huidige simulatiesnelheid. Deze analyse is dus een gevoeligheidsanalyse, geen volledig nieuwe inverse dynamica.

In [ ]:
# ============================================================
# Gevoeligheid aan wandelsnelheid
# ============================================================

# ------------------------------------------------------------
# Basisdata uit bestaande inverse dynamica
# ------------------------------------------------------------

omega_driver = np.mean(np.abs(dtheta_2))  # [rad/s]
i_end_cycle = int(np.round((2*np.pi / omega_driver) / Ts))

t_motor = t[:i_end_cycle] - t[0]
theta_motor = -theta_2[:i_end_cycle]
theta_motor_norm = (theta_motor - theta_motor[0]) % (2*np.pi)

M_without_GRF = M_0[:i_end_cycle]       # [Nm]
M_with_GRF = M_0_wf[:i_end_cycle]       # [Nm]
omega_motor = dtheta_2[:i_end_cycle]    # [rad/s]

# Decompositie voor snelheidsgevoeligheid
M_inertia_ref = M_without_GRF
M_ground_ref = M_with_GRF - M_without_GRF

# ------------------------------------------------------------
# Referentie uit paper
# ------------------------------------------------------------

v_ref_kmh = 1.5      # [km/h]
rpm_ref = 20.0       # [rpm]

def kmh_to_rpm(v_kmh):
    return rpm_ref * v_kmh / v_ref_kmh

def rpm_to_kmh(rpm):
    return v_ref_kmh * rpm / rpm_ref

def rpm_to_omega(rpm):
    return rpm * 2*np.pi / 60

# Afstand per cyclus/inputomwenteling
distance_per_cycle = (v_ref_kmh / 3.6) / (rpm_ref / 60)  # [m/cycle]

# Huidige simulatiesnelheid
rpm_model = omega_driver * 60 / (2*np.pi)

# Rendement gekozen DC-servo + planetaire gearbox
eta_motor_controller = 0.80
eta_gearbox = 0.94
eta_total = eta_motor_controller * eta_gearbox

speed_cases = [1.0, 2.0, 3.0]

speed_results = []

plt.figure(figsize=(14, 4))

# ------------------------------------------------------------
# Grafiek 1: koppel
# ------------------------------------------------------------

plt.subplot(1, 2, 1)

for v_kmh in speed_cases:
    rpm_case = kmh_to_rpm(v_kmh)
    k = rpm_case / rpm_model
    
    M_case = k**2 * M_inertia_ref + M_ground_ref
    omega_case = k * omega_motor
    t_case = t_motor / k
    
    P_case = -M_case * omega_case
    
    T_case = 60 / rpm_case
    E_pos_cycle = np.trapezoid(np.maximum(P_case, 0), t_case)
    E_abs_cycle = np.trapezoid(np.abs(P_case), t_case)
    
    P_mean_pos = E_pos_cycle / T_case
    P_mean_abs = E_abs_cycle / T_case
    P_peak_abs = np.max(np.abs(P_case))
    
    E_mech_per_meter = E_pos_cycle / distance_per_cycle
    P_elec_mean = P_mean_pos / eta_total
    E_elec_per_hour_Wh = P_elec_mean
    E_elec_per_hour_kWh = E_elec_per_hour_Wh / 1000
    
    speed_results.append({
        "wandelsnelheid [km/h]": v_kmh,
        "input-rpm [rpm]": rpm_case,
        "hoeksnelheid gemiddeld [rad/s]": rpm_to_omega(rpm_case),
        "snelheidsfactor k [-]": k,
        "max |M| [Nm]": np.max(np.abs(M_case)),
        "RMS-koppel [Nm]": np.sqrt(np.mean(M_case**2)),
        "max |P| [W]": P_peak_abs,
        "gemiddeld positief P_mech [W]": P_mean_pos,
        "energie per cyclus [J]": E_pos_cycle,
        "energie per meter [J/m]": E_mech_per_meter,
        "geschat P_elec gemiddeld [W]": P_elec_mean,
        "elektrische energie per uur [Wh]": E_elec_per_hour_Wh,
        "elektrische energie per uur [kWh]": E_elec_per_hour_kWh,
    })
    
    plt.plot(theta_motor_norm, M_case, label=f"{v_kmh:.1f} km/h")

plt.xlabel(r"inputhoek $\theta_2$ [rad]")
plt.ylabel("aandrijfmoment [Nm]")
plt.title("Effect van wandelsnelheid op koppel")
plt.grid()
plt.legend()

# ------------------------------------------------------------
# Grafiek 2: vermogen
# ------------------------------------------------------------

plt.subplot(1, 2, 2)

for v_kmh in speed_cases:
    rpm_case = kmh_to_rpm(v_kmh)
    k = rpm_case / rpm_model
    
    M_case = k**2 * M_inertia_ref + M_ground_ref
    omega_case = k * omega_motor
    P_case = -M_case * omega_case
    
    plt.plot(theta_motor_norm, P_case, label=f"{v_kmh:.1f} km/h")

plt.xlabel(r"inputhoek $\theta_2$ [rad]")
plt.ylabel("mechanisch vermogen [W]")
plt.title("Effect van wandelsnelheid op vermogen")
plt.grid()
plt.legend()

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Samenvattende tabel
# ------------------------------------------------------------

speed_results = pd.DataFrame(speed_results)

display(speed_results.round(3))

### Batterij-inschatting bij maximale snelheid

Omdat de gekozen motor een DC-servo is, is batterijvoeding mogelijk. We schatten daarom de batterijgrootte voor werking bij de maximale gewenste snelheid van 3 km/h.

We gebruiken het geschatte elektrische vermogen uit de snelheidsanalyse en nemen een marge op de batterijcapaciteit. Die marge houdt rekening met:
- verliezen die niet in het model zitten;
- het feit dat een batterij niet volledig ontladen wordt;
- piekstromen;
- onzekerheid op het rendement.

Voor een ruwe schatting nemen we:
- batterijspanning: 24 V;
- capaciteitsmarge: 30%;
- energiedichtheid batterijpakket: 150 Wh/kg;
- batterijkost: 0,40 €/Wh.

In [ ]:
# ============================================================
# Batterij-inschatting bij hoogste snelheid
# ============================================================

# Batterijparameters
battery_voltage = 24.0          # [V]
battery_margin = 1.30           # [-]
battery_energy_density = 150.0  # [Wh/kg], ruwe schatting voor compleet pakket
battery_cost_per_Wh = 0.40      # [EUR/Wh], ruwe schatting

# Neem hoogste snelheid uit de vorige tabel
highest_speed = speed_results.loc[
    speed_results["wandelsnelheid [km/h]"].idxmax()
]

P_elec_avg_high = highest_speed["geschat P_elec gemiddeld [W]"]

# Piekvermogen opnieuw bepalen voor hoogste snelheid
v_high = highest_speed["wandelsnelheid [km/h]"]
rpm_high = kmh_to_rpm(v_high)
k_high = rpm_high / rpm_model

M_high = k_high**2 * M_inertia_ref + M_ground_ref
omega_high = k_high * omega_motor
P_high = -M_high * omega_high

P_elec_peak_high = np.max(np.maximum(P_high, 0)) / eta_total

I_avg = P_elec_avg_high / battery_voltage
I_peak = P_elec_peak_high / battery_voltage

runtime_minutes = [10, 20, 30]

battery_results = []

for runtime_min in runtime_minutes:
    runtime_h = runtime_min / 60
    
    E_without_margin = P_elec_avg_high * runtime_h
    E_with_margin = E_without_margin * battery_margin
    
    capacity_Ah = E_with_margin / battery_voltage
    battery_mass = E_with_margin / battery_energy_density
    battery_cost = E_with_margin * battery_cost_per_Wh
    
    battery_results.append({
        "gebruikstijd [min]": runtime_min,
        "energie zonder marge [Wh]": E_without_margin,
        "energie met marge [Wh]": E_with_margin,
        "capaciteit bij 24 V [Ah]": capacity_Ah,
        "gemiddelde stroom [A]": I_avg,
        "geschatte piekstroom [A]": I_peak,
        "geschat batterijgewicht [kg]": battery_mass,
        "geschatte batterijkost [EUR]": battery_cost,
    })

battery_results = pd.DataFrame(battery_results)

print(f"Batterij-inschatting bij {v_high:.1f} km/h")
print(f"Gemiddeld elektrisch vermogen: {P_elec_avg_high:.3f} W")
print(f"Geschat elektrisch piekvermogen: {P_elec_peak_high:.3f} W")

display(battery_results.round(3))

De batterijcapaciteit blijft in deze schatting relatief beperkt. Dat komt doordat het gemiddelde mechanische vermogen van het mechanisme vrij laag is. In de praktijk zal het echte verbruik hoger liggen door extra wrijving, controllerverliezen en niet-ideale werking van de motor bij lage snelheid.

Voor de praktische batterijselectie is niet alleen de energiecapaciteit belangrijk, maar ook de maximaal toelaatbare stroom. De batterij moet de piekstromen tijdens de cyclus kunnen leveren zonder te grote spanningsval.

### Controle op dode punten en transmissiehoeken

In de les werd een dood punt besproken aan de hand van een vierstangenmechanisme. Een dood punt ontstaat wanneer de aandrijvende kracht of beweging in een ongunstige richting werkt, waardoor het mechanisme moeilijk of niet verder beweegt. Dit treedt vooral op wanneer staven bijna collineair worden.

Voor een vierstangenmechanisme wordt dit vaak beoordeeld met de transmissiehoek $\gamma$. Dit is de hoek tussen de koppelstang en de uitvoerstaaf. Een transmissiehoek dicht bij $90^\circ$ is gunstig, omdat het aandrijfkoppel dan efficiënt wordt doorgegeven. Een transmissiehoek dicht bij $0^\circ$ of $180^\circ$ is ongunstig en wijst op een mogelijke dode-puntconfiguratie.

Ons mechanisme bestaat uit drie gekoppelde gesloten lussen. Daarom is er niet één transmissiehoek voor het volledige mechanisme. We bekijken voor de eesrte twee lussen een transmissieachtige hoek:

- lus 1: hoek tussen $r_3$ en $r_4$
- lus 2: hoek tussen $r_6$ en $L_1$

<p align="center">
  <img src="Images\transmissiehoek.png" width="700"><br>
  <b>Figuur 1:</b> Eigen schets van transmissiehoeken in lus 1 en lus 2
</p>

Deze hoeken worden berekend uit de hoeken die eerder uit de sluitingsvergelijkingen werden gevonden. We bekijken ook de afstand tot het dichtstbijzijnde dode punt:

$\Delta \gamma = \min(\gamma, 180^\circ - \gamma)$

Hoe groter deze marge, hoe verder het mechanisme van een collineaire configuratie zit.

Daarnaast bekijken we het conditiegetal van de snelheidsmatrices. Dit sluit aan bij dezelfde redenering: wanneer twee relevante staven bijna collineair worden, wordt de snelheidsmatrix bijna singulier. Het conditiegetal stijgt dan sterk. Het conditiegetal gebruiken we dus als numerieke controle naast de fysisch intuïtieve transmissiehoeken.

Wikipedia: "The Grashof condition for a four-bar linkage states: If the sum of the shortest and longest link of a planar quadrilateral linkage is less than or equal to the sum of the remaining two links, then the shortest link can rotate fully with respect to a neighboring link"

In [ ]:
# ============================================================
# Controle op dode punten via transmissiehoeken
# ============================================================

# Eén volledige cyclus selecteren
omega_driver = np.mean(np.abs(dtheta_2))  # [rad/s]
i_end_cycle = int(np.round((2*np.pi / omega_driver) / Ts))

t_dp = t[:i_end_cycle] - t[0]
theta_input = (-theta_2[:i_end_cycle] + theta_2[0]) % (2*np.pi)

M_driver = M_0_wf[:i_end_cycle]

# ------------------------------------------------------------
# Hulpfuncties
# ------------------------------------------------------------

def angle_between_links(theta_a, theta_b):
    """
    Berekent de kleinste hoek tussen twee staven.
    Resultaat ligt tussen 0 en pi radialen.
    """
    diff = (theta_a - theta_b + np.pi) % (2*np.pi) - np.pi
    return np.abs(diff)

def rad_to_deg(angle_rad):
    return angle_rad * 180 / np.pi

# ------------------------------------------------------------
# Transmissieachtige hoeken per lus
# ------------------------------------------------------------

# Lus 1: vierstangenlus O2-A-B-D
# transmissiehoek tussen koppelstang r3 en uitvoerstaaf r4
gamma_1 = angle_between_links(theta_3[:i_end_cycle], theta_4[:i_end_cycle])

# Lus 2: lus O2-A-E-D
# hoek tussen r6 en L1
gamma_2 = angle_between_links(theta_6[:i_end_cycle], theta_f[:i_end_cycle])

gamma_list = [gamma_1, gamma_2]
gamma_names = ["lus 1: r3-r4", "lus 2: r6-L1"]

# Marge tot dichtstbijzijnde dode punt
# 0 graden of 180 graden zijn ongunstig
margin_1 = np.minimum(gamma_1, np.pi - gamma_1)
margin_2 = np.minimum(gamma_2, np.pi - gamma_2)

margin_list = [margin_1, margin_2]

# ------------------------------------------------------------
# Samenvatting per lus
# ------------------------------------------------------------

summary_dead_points = []

for name, gamma, margin in zip(gamma_names, gamma_list, margin_list):
    i_worst = np.argmin(margin)
    
    summary_dead_points.append({
        "lus": name,
        "min transmissiehoek [deg]": np.min(rad_to_deg(gamma)),
        "max transmissiehoek [deg]": np.max(rad_to_deg(gamma)),
        "theta bij kleinste marge [rad]": theta_input[i_worst],
        "|M_0,wf| bij kleinste marge [Nm]": np.abs(M_driver[i_worst]),
    })

summary_dead_points = pd.DataFrame(summary_dead_points)

display(summary_dead_points.round(3))

In [ ]:
# ============================================================
# Grashof-check voor de eerste vierstangenlus
# ============================================================

links_loop1 = np.array([r_1, r_2, r_3, r_4])
link_names = np.array(["r1", "r2", "r3", "r4"])

order = np.argsort(links_loop1)
sorted_links = links_loop1[order]
sorted_names = link_names[order]

s = sorted_links[0]   # kortste
l = sorted_links[-1]  # langste
p = sorted_links[1]
q = sorted_links[2]

grashof_value_left = s + l
grashof_value_right = p + q

print("Grashof-check voor lus 1")
print("------------------------")
print(f"kortste staaf: {sorted_names[0]} = {s:.3f} m")
print(f"langste staaf: {sorted_names[-1]} = {l:.3f} m")
print(f"s + l = {grashof_value_left:.3f} m")
print(f"p + q = {grashof_value_right:.3f} m")

if grashof_value_left <= grashof_value_right:
    print("Conclusie: lus 1 voldoet aan de Grashof-voorwaarde.")
else:
    print("Conclusie: lus 1 voldoet niet aan de Grashof-voorwaarde.")

if sorted_names[0] == "r2":
    print("De inputlink r2 is de kortste staaf. Dit ondersteunt dat de inputlink volledig kan roteren.")
else:
    print("De inputlink r2 is niet de kortste staaf. Controleer of volledige rotatie van de inputlink mogelijk blijft.")

In [ ]:
# ============================================================
# Plot transmissiehoeken
# ============================================================

plt.figure(figsize=(8, 4))

plt.plot(theta_input, rad_to_deg(gamma_1), label=r"lus 1: $\gamma_1$ tussen $r_3$ en $r_4$")
plt.plot(theta_input, rad_to_deg(gamma_2), label=r"lus 2: $\gamma_2$ tussen $r_6$ en $L_1$")

plt.axhline(90, linestyle="--", label=r"ideaal: $90^\circ$")
plt.axhline(30, linestyle=":", label=r"ongunstige zone")
plt.axhline(150, linestyle=":")

plt.xlabel(r"inputhoek $\theta_2$ [rad]")
plt.ylabel("transmissieachtige hoek [deg]")
plt.title("Transmissiehoeken van de eerste twee lussen")
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Vergelijking: marge tot dood punt en motorkoppel
# ============================================================

min_margin_all = np.minimum(margin_1, margin_2)
i_worst_margin = np.argmin(min_margin_all)
i_peak_torque = np.argmax(np.abs(M_driver))

# Motorkoppel en slechtste transmissiemarge samen
fig, ax1 = plt.subplots(figsize=(8, 4))

ax1.plot(theta_input, np.abs(M_driver), label=r"$|M_{0,wf}|$")
ax1.axvline(theta_input[i_peak_torque], linestyle=":", label="max |koppel|")
ax1.set_xlabel(r"inputhoek $\theta_2$ [rad]")
ax1.set_ylabel("absoluut aandrijfmoment [Nm]")
ax1.grid()

ax2 = ax1.twinx()
ax2.plot(theta_input, rad_to_deg(min_margin_all), linestyle="--", label="kleinste transmissiemarge")
ax2.axvline(theta_input[i_worst_margin], linestyle="--", label="kleinste marge")
ax2.set_ylabel("kleinste marge tot dood punt [deg]")

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="best")

plt.title("Motorkoppel vergeleken met nabijheid van dode punten")
plt.tight_layout()
plt.show()

# Samenvatting van kritieke posities
dead_point_comparison = pd.DataFrame([{
    "kleinste marge tot dood punt [deg]": rad_to_deg(min_margin_all[i_worst_margin]),
    "theta bij kleinste marge [rad]": theta_input[i_worst_margin],
    "|M_0,wf| bij kleinste marge [Nm]": np.abs(M_driver[i_worst_margin]),
    "max |M_0,wf| [Nm]": np.abs(M_driver[i_peak_torque]),
    "theta bij max |M_0,wf| [rad]": theta_input[i_peak_torque],
    "hoekafstand tussen beide [rad]": min(
        abs(theta_input[i_worst_margin] - theta_input[i_peak_torque]),
        2*np.pi - abs(theta_input[i_worst_margin] - theta_input[i_peak_torque])
    )
}])

display(dead_point_comparison.round(3))

In [ ]:
# ============================================================
# Numerieke controle via condition number
# ============================================================

cond_cycle = cond[:i_end_cycle, :]

plt.figure(figsize=(8, 4))

plt.plot(theta_input, cond_cycle[:, 0], label="lus 1")
plt.plot(theta_input, cond_cycle[:, 1], label="lus 2")
plt.plot(theta_input, cond_cycle[:, 2], label="lus 3")

plt.xlabel(r"inputhoek $\theta_2$ [rad]")
plt.ylabel("condition number [-]")
plt.title("Condition number van de snelheidsmatrices")
plt.yscale("log")
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

cond_max = np.max(cond_cycle, axis=1)
i_worst_cond = np.argmax(cond_max)

condition_summary = pd.DataFrame([{
    "max condition number [-]": cond_max[i_worst_cond],
    "theta bij max condition number [rad]": theta_input[i_worst_cond],
    "|M_0,wf| bij max condition number [Nm]": np.abs(M_driver[i_worst_cond]),
    "kleinste marge bij max condition number [deg]": rad_to_deg(min_margin_all[i_worst_cond]),
}])

display(condition_summary.round(3))

### Interpretatie

De transmissiehoeken sluiten rechtstreeks aan bij de methode uit de les. Een transmissiehoek dicht bij $90^\circ$ is gunstig, terwijl een hoek dicht bij $0^\circ$ of $180^\circ$ wijst op een slechte krachtoverdracht of een mogelijke dode-puntconfiguratie.

De belangrijkste controle is of een kleine marge samenvalt met een hoog gevraagd motorkoppel $M_{0,wf}$. Als dat het geval is, moet de motor veel koppel leveren in een mechanisch ongunstige configuratie. Dat zou kritisch zijn voor de motorselectie, gewrichtskrachten en gevoeligheid voor speling.

Het conditiegetal gebruiken we als numerieke bevestiging van hetzelfde principe. Wanneer de transmissiehoek slecht wordt, worden de snelheidsvergelijkingen slecht geconditioneerd. Een hoge condition number wijst dus op een configuratie dicht bij een singulariteit. Een hoog conditiegetal betekent dat kleine fouten in posities, lengtes of snelheden een veel grotere fout kunnen geven in de berekende hoeksnelheden of krachten. We zien dat lus 3 een veel hoger conditiegetal heeft, wat ook logisch is als je kijkt naar de beweging. Dit komt omdat het conditiegetal zowel afhankelijk is van de transmissiehoek als de verhouding van de lengtes van de staven. In lus 3 is r7 significant groter dan r8 wat leidt tot een groter conditiegetal.

<div style="display:flex; align-items:flex-start; gap:30px;">

<div style="flex:1.4;">

<img src="Images\Ring_Pinion_2.gif" width="600"><br>
<b>Figure 1:</b> Transmissie van motor op aandrijfas beiden benen

</div>


<img src="Images\fazeverschuiving.gif" width="400"><br>
<b>Figure 2:</b> Krukas met 180° mechanische faseverschuiving (voorbeeld fiets)

</div>

</div>

## Twee benen aandrijven met één motor

Tot nu toe werd de motor gedimensioneerd voor één been. Omdat een wandelbeweging links en rechts 180° uit fase is, onderzoeken we nu of beide benen mechanisch door één centrale motor kunnen worden aangedreven.

Voor twee identieke mechanismen met 180° faseverschil wordt het totale koppel:

$M_\text{benen}(\theta)=M_{0,wf}(\theta) + M_{0,wf}(\theta + \pi)$

Omdat de verbinding van de centrale motor naar de twee benen niet ideaal is, nemen we een transmissierendement aan:

$\eta_\text{transmissie} = 0{,}9$

Het koppel dat de motor moet leveren wordt dan:

$M_\text{motor,2benen}(\theta)=\frac{M_{0,wf}(\theta) + M_{0,wf}(\theta + \pi)}{\eta_\text{transmissie}}$

Deze analyse is een eerste schatting. We nemen aan dat beide benen identiek zijn en exact 180° uit fase bewegen.

### Mechanische koppeling

Een mogelijke centrale aandrijving bestaat uit een motoruitgang die via een pinion/crown-wheel-principe een starre dwarsas aandrijft. Daarna worden links en rechts crankarmen op de dwarsas geplaatst met een onderlinge montagehoek van 180°, zoals bij fietspedalen.


<div style="display: flex; gap: 25px; align-items: flex-start;">
  <figure style="width: 48%; text-align: center;">
    <img src="Images/Ring_Pinion_2.gif" style="width: 100%;">
    <figcaption><b>Figuur:</b> centrale motoroverbrenging naar dwarsas.</figcaption>
  </figure>

  <figure style="width: 32%; text-align: center;">
    <img src="Images/fazeverschuiving.gif" style="width: 100%;">
    <figcaption><b>Figuur:</b> twee cranks 180° uit fase, zoals bij fietspedalen.</figcaption>
  </figure>
</div>

In [ ]:
# ============================================================
# Twee benen met één centrale motor
# ============================================================

# ------------------------------------------------------------
# Periodiek signaal 180° verschuiven
# ------------------------------------------------------------

def periodic_shift(y, phase_rad):
    """
    Verschuift een periodiek signaal y over phase_rad.
    Eén volledige cyclus wordt verondersteld.
    """
    N = len(y)
    x = np.arange(N)
    shift_samples = phase_rad / (2*np.pi) * N
    x_query = (x + shift_samples) % N
    
    return np.interp(x_query, np.r_[x, N], np.r_[y, y[0]])

def find_peak_indices(y, n_peaks=2, min_separation_fraction=0.20):
    """
    Zoekt de n grootste lokale maxima van |y|.
    De minimumafstand vermijdt dat dezelfde piek meerdere keren wordt gekozen.
    """
    y_abs = np.abs(y)
    N = len(y_abs)
    min_separation = int(min_separation_fraction * N)
    
    candidates = []
    
    for i in range(N):
        left = y_abs[i-1]
        middle = y_abs[i]
        right = y_abs[(i+1) % N]
        
        if middle >= left and middle >= right:
            candidates.append(i)
    
    candidates = sorted(candidates, key=lambda i: y_abs[i], reverse=True)
    
    selected = []
    
    for i in candidates:
        far_enough = True
        
        for j in selected:
            distance = abs(i - j)
            circular_distance = min(distance, N - distance)
            
            if circular_distance < min_separation:
                far_enough = False
        
        if far_enough:
            selected.append(i)
        
        if len(selected) == n_peaks:
            break
    
    return sorted(selected)

def signal_summary(name, M, P, time):
    """
    Samenvatting van koppel, vermogen en energie over één cyclus.
    """
    E_net = np.trapezoid(P, time)
    E_abs = np.trapezoid(np.abs(P), time)
    E_pos = np.trapezoid(np.maximum(P, 0), time)
    
    return {
        "case": name,
        "max |M| [Nm]": np.max(np.abs(M)),
        "RMS-koppel [Nm]": np.sqrt(np.mean(M**2)),
        "max |P| [W]": np.max(np.abs(P)),
        "gemiddeld |P| [W]": E_abs / T_cycle,
        "gemiddeld positief P [W]": E_pos / T_cycle,
        "netto energie/cyclus [J]": E_net,
        "absolute energie/cyclus [J]": E_abs,
        "positieve energie/cyclus [J]": E_pos,
    }

# ------------------------------------------------------------
# Twee benen 180° uit fase
# ------------------------------------------------------------

M_left = M_one_leg
M_right = periodic_shift(M_one_leg, np.pi)

eta_transmission = 0.90

M_2legs_at_mechanisms = M_left + M_right
M_motor_2legs = M_2legs_at_mechanisms / eta_transmission

# Zelfde tekenconventie als eerder: P = -M * omega
P_one_leg = -M_one_leg * omega_input
P_left = -M_left * omega_input
P_right = -M_right * omega_input
P_motor_2legs = -M_motor_2legs * omega_input

# Maxima van het tweebenige koppelprofiel
peak_indices_2legs = find_peak_indices(M_motor_2legs, n_peaks=2)
# ============================================================
# Twee benen met één centrale motor
# ============================================================

# ------------------------------------------------------------
# Periodiek signaal 180° verschuiven
# ------------------------------------------------------------

def periodic_shift(y, phase_rad):
    """
    Verschuift een periodiek signaal y over phase_rad.
    Eén volledige cyclus wordt verondersteld.
    """
    N = len(y)
    x = np.arange(N)
    shift_samples = phase_rad / (2*np.pi) * N
    x_query = (x + shift_samples) % N
    
    return np.interp(x_query, np.r_[x, N], np.r_[y, y[0]])

# Tweede been: identiek koppelprofiel, maar 180° uit fase
M_left = M_one_leg
M_right = periodic_shift(M_one_leg, np.pi)

# Transmissierendement van centrale motor naar beide benen
eta_transmission = 0.90

# Totaalkoppel aan de centrale motor
M_2legs_at_mechanisms = M_left + M_right
M_motor_2legs = M_2legs_at_mechanisms / eta_transmission

# Vermogen
# Zelfde tekenconventie als eerder: P = -M * omega
P_left = -M_left * omega_input
P_right = -M_right * omega_input
P_2legs_at_mechanisms = P_left + P_right
P_motor_2legs = -M_motor_2legs * omega_input

# ------------------------------------------------------------
# Kengetallen
# ------------------------------------------------------------

i_peak_2legs = np.argmax(np.abs(M_motor_2legs))

M_peak_2legs = np.max(np.abs(M_motor_2legs))
M_rms_2legs = np.sqrt(np.mean(M_motor_2legs**2))

P_peak_2legs = np.max(np.abs(P_motor_2legs))
E_net_2legs = np.trapezoid(P_motor_2legs, t_2leg)
E_abs_2legs = np.trapezoid(np.abs(P_motor_2legs), t_2leg)
E_pos_2legs = np.trapezoid(np.maximum(P_motor_2legs, 0), t_2leg)

P_mean_abs_2legs = E_abs_2legs / T_cycle
P_mean_pos_2legs = E_pos_2legs / T_cycle

two_leg_requirements = pd.DataFrame([{
    "case": "1 centrale motor voor 2 benen",
    "transmissierendement [-]": eta_transmission,
    "max |M| [Nm]": M_peak_2legs,
    "RMS-koppel [Nm]": M_rms_2legs,
    "max |P| [W]": P_peak_2legs,
    "gemiddeld |P| [W]": P_mean_abs_2legs,
    "netto energie/cyclus [J]": E_net_2legs,
}])

display(two_leg_requirements.round(3))


In [ ]:
# ============================================================
# Grafiek koppel voor twee benen
# ============================================================

plt.figure(figsize=(8, 4))

# Individuele benen subtieler
plt.plot(theta_2leg, M_left, linewidth=1, alpha=0.30, label="linker been")
plt.plot(theta_2leg, M_right, linewidth=1, alpha=0.30, label="rechter been, 180° verschoven")

# Totaal duidelijker
plt.plot(theta_2leg, M_motor_2legs, linewidth=2.5, label="totaal aan centrale motor incl. verliezen")

# Beide maxima aanduiden
for count, i_peak in enumerate(peak_indices_2legs):
    label_line = "max |motorkoppel|" if count == 0 else None
    plt.axvline(theta_2leg[i_peak], linestyle="--", alpha=0.8, label=label_line)
    plt.scatter(theta_2leg[i_peak], M_motor_2legs[i_peak], zorder=3)

plt.xlabel(r"inputhoek $\theta_2$ [rad]")
plt.ylabel("aandrijfmoment [Nm]")
plt.title("Motorkoppel voor twee benen met 180° faseverschil")
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

print("Piekwaarden voor centrale motor:")
for i_peak in peak_indices_2legs:
    print(f"theta = {theta_2leg[i_peak]:.3f} rad, M = {M_motor_2legs[i_peak]:.3f} Nm")

print()
print(f"Maximaal absoluut motorkoppel: {M_peak_2legs:.3f} Nm")
print(f"Maximaal absoluut motorvermogen: {P_peak_2legs:.3f} W")

### Energievergelijking

Voor twee aparte motoren nemen we geen regeneratie aan. Elke motor moet dus zijn eigen positieve mechanische arbeid leveren. De totale positieve energie is dan $2$ keer de positieve energie van één been.

Bij één centrale motor worden de twee vermogensprofielen eerst mechanisch samengevoegd. Daardoor kan negatief vermogen van het ene been gedeeltelijk samenvallen met positief vermogen van het andere been. Dit kan het totale positieve vermogen aan de centrale motor verlagen. Daartegenover staat wel het extra transmissieverlies van de centrale koppeling, waarvoor we $\eta = 0{,}90$ aannemen.

Als regeneratie of een gedeelde DC-bus tussen twee motorcontrollers mogelijk zou zijn, kan het verschil tussen één centrale motor en twee aparte motoren kleiner worden. In deze analyse nemen we echter geen regeneratie aan.

Voor het energieverbruik is een rechtstreekse vergelijking met één been alleen eerlijk wanneer we de tweebenige resultaten per been normaliseren. Daarom worden het gemiddeld positief vermogen en de positieve energie per cyclus ook per been gerapporteerd. De totale energie van de centrale motor blijft wel relevant voor de batterij en het totale systeemverbruik. De per-beenwaarden zijn vooral nuttig om te beoordelen of één centrale motor energetisch gunstiger of ongunstiger is dan twee aparte motoren.

In [ ]:
# ============================================================
# Energievergelijking: twee aparte motoren versus één centrale motor
# ============================================================

# Twee aparte motoren zonder regeneratie:
# elke motor moet zijn eigen positieve vermogen leveren.
# Omdat de benen identiek zijn, is dit 2 keer de positieve energie van één been.
P_mean_pos_two_separate_motors_mech = 2 * P_mean_pos_one_leg
E_pos_two_separate_motors_mech = 2 * E_pos_one_leg

# Eén centrale motor:
# positieve energie van het gecombineerde profiel, inclusief transmissieverlies.
P_mean_pos_one_central_motor_mech = P_mean_pos_2legs
E_pos_one_central_motor_mech = E_pos_2legs

energy_comparison = pd.DataFrame([
    {
        "configuratie": "2 aparte motoren",
        "mechanisch model": "2 × positief vermogen van één been",
        "gemiddeld positief P_mech [W]": P_mean_pos_two_separate_motors_mech,
        "positieve energie/cyclus [J]": E_pos_two_separate_motors_mech,
        "factor t.o.v. 1 been [-]": E_pos_two_separate_motors_mech / E_pos_one_leg,
    },
    {
        "configuratie": "1 centrale motor",
        "mechanisch model": "gecombineerd profiel, 180° fase, η = 0.90",
        "gemiddeld positief P_mech [W]": P_mean_pos_one_central_motor_mech,
        "positieve energie/cyclus [J]": E_pos_one_central_motor_mech,
        "factor t.o.v. 1 been [-]": E_pos_one_central_motor_mech / E_pos_one_leg,
    },
])

display(energy_comparison.round(3))

In [ ]:
# ============================================================
# Check zelfde DC-motor voor centrale aandrijving van twee benen
# ============================================================

safety_factor = 1.5

# Specificaties van eerder gekozen DC-servo + planetaire gearbox
# DSY 200 W DC-servo + HTG60-G50 50:1 gearbox
dc_motor = {
    "naam": "200 W DC-servo + 50:1 planetaire gearbox",
    "continu koppel output [Nm]": 29.939,
    "start-/piekkoppel output [Nm]": 89.770,
    "snelste rpm continu [rpm]": 60.000,
    "gewicht [kg]": 3.810,
    "kost [EUR]": 185.980,
}

required_peak_2legs_with_SF = safety_factor * M_peak_2legs

motor_check_2legs = pd.DataFrame([{
    "motor": dc_motor["naam"],
    "vereist piekkoppel incl. SF [Nm]": required_peak_2legs_with_SF,
    "beschikbaar piekkoppel [Nm]": dc_motor["start-/piekkoppel output [Nm]"],
    "piekmarge [-]": dc_motor["start-/piekkoppel output [Nm]"] / required_peak_2legs_with_SF,
    "vereist RMS-koppel [Nm]": M_rms_2legs,
    "beschikbaar continu koppel [Nm]": dc_motor["continu koppel output [Nm]"],
    "RMS-marge [-]": dc_motor["continu koppel output [Nm]"] / M_rms_2legs,
    "max rpm motoroutput [rpm]": dc_motor["snelste rpm continu [rpm]"],
}])

display(motor_check_2legs.round(3))

if dc_motor["start-/piekkoppel output [Nm]"] >= required_peak_2legs_with_SF and dc_motor["continu koppel output [Nm]"] >= M_rms_2legs:
    print("Conclusie: dezelfde DC-motorconfiguratie voldoet ook voor één centrale motor die beide benen aandrijft.")
else:
    print("Conclusie: dezelfde DC-motorconfiguratie voldoet niet. Een zwaardere DC-servo of grotere reductie is nodig.")

In [ ]:
# ============================================================
# Kwantitatieve vergelijking: 1 centrale motor vs 2 aparte motoren
# ============================================================

electricity_price = 0.107  # [EUR/kWh]

# Efficiëntie van de DC-servo + planetaire gearbox
eta_motor_controller = 0.80
eta_planetary = 0.94
eta_dc_total = eta_motor_controller * eta_planetary

# Extra onderdelen voor centrale transmissie naar beide benen
extra_transmission_mass = 1.000    # [kg], ruwe schatting
extra_transmission_cost = 100.000  # [EUR], ruwe schatting

# Specificaties gekozen DC-servo + gearbox
dc_motor = {
    "naam": "200 W DC-servo + 50:1 planetaire gearbox",
    "continu koppel output [Nm]": 29.939,
    "start-/piekkoppel output [Nm]": 89.770,
    "snelste rpm continu [rpm]": 60.000,
    "gewicht [kg]": 3.810,
    "kost [EUR]": 185.980,
}

# ------------------------------------------------------------
# Twee aparte motoren
# ------------------------------------------------------------

# Expliciet 2 keer het positieve vermogen van één been
P_elec_two_separate_motors = (2 * P_mean_pos_one_leg) / eta_dc_total

# ------------------------------------------------------------
# Eén centrale motor
# ------------------------------------------------------------

# Gecombineerd positief vermogen van beide benen, inclusief centrale transmissieverlies
P_elec_one_central_motor = P_mean_pos_2legs / eta_dc_total

comparison_1_vs_2 = pd.DataFrame([
    {
        "configuratie": "2 aparte motoren",
        "aantal motoren": 2,
        "motoren + transmissie massa [kg]": 2 * dc_motor["gewicht [kg]"],
        "motoren + transmissie kost [EUR]": 2 * dc_motor["kost [EUR]"],
        "max koppel per motor [Nm]": M_peak_one_leg,
        "RMS-koppel per motor [Nm]": M_rms_one_leg,
        "gemiddeld elektrisch vermogen totaal [W]": P_elec_two_separate_motors,
        "energie per uur [kWh]": P_elec_two_separate_motors / 1000,
        "energiekost per 100 uur [EUR]": 100 * P_elec_two_separate_motors / 1000 * electricity_price,
    },
    {
        "configuratie": "1 centrale motor + 180° transmissie",
        "aantal motoren": 1,
        "motoren + transmissie massa [kg]": dc_motor["gewicht [kg]"] + extra_transmission_mass,
        "motoren + transmissie kost [EUR]": dc_motor["kost [EUR]"] + extra_transmission_cost,
        "max koppel per motor [Nm]": M_peak_2legs,
        "RMS-koppel per motor [Nm]": M_rms_2legs,
        "gemiddeld elektrisch vermogen totaal [W]": P_elec_one_central_motor,
        "energie per uur [kWh]": P_elec_one_central_motor / 1000,
        "energiekost per 100 uur [EUR]": 100 * P_elec_one_central_motor / 1000 * electricity_price,
    }
])

display(comparison_1_vs_2.round(3))

### Kwalitatieve vergelijking: één motor of twee motoren

**Eén centrale motor voordelen:**
- mechanisch gegarandeerd 180° faseverschil
- slechts één motor en één controller
- massa kan centraal geplaatst worden
- gladder koppelprofiel door faseverschuiving
- mechanische energie-uitwisseling tussen benen mogelijk
- lichter (aanname gewicht transmissie 1kg)
- goedkoper investeringskost (aanname kost transmissie €100)

**Eén centrale motor nadelen:**
- extra transmissie nodig
- starten en stoppen met beide voeten naast elkaar is onmogelijk
- schokken aan één been worden doorgegeven aan het andere been

**Twee motoren voordelen:**
- geschikt voor asymmetrische bewegingen
- makkelijker om kleine verschillen tussen benen te compenseren
- zuiger voor energieverbruik

**Twee motoren nadelen:**
- twee motoren en twee controllers nodig
- duurder en zwaarder
- softwarematige synchronisatie nodig om 180° faseverschil te behouden

Voor zuiver periodisch wandelen is één centrale motor mechanisch aantrekkelijk. Voor een echt draagbaar revalidatie-exoskelet zijn twee motoren praktischer en flexibeler.